# Stylized Simulation

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, asdict
from typing import Dict, Any, Optional, List


# ===================================================================
# 0. Utilities
# ===================================================================

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


# ===================================================================
# 1. Simulation configuration
# ===================================================================

@dataclass
class SimulationConfig:
    # basic structure
    n_clients: int = 100               # total number of clients
    T: int = 300                       # number of rounds
    gaming_frac: float = 0.3           # fraction of clients using gaming strategy

    # reward-related
    base_reward: float = 1.0           # scale from metric to reward
    reward_bias: float = 0.2           # baseline reward independent of metric

    # audit / penalty
    audit_budget: int = 5              # maximum number of clients audited per round
    alpha_penalty: float = 0.7         # penalty strength α when caught
    p_detect: float = 0.7              # probability of detecting gaming when audited

    # public / private metric mix
    rho_pub: float = 0.6               # weight on public metric (PB vs PC mix)
    noise_pub: float = 0.01            # noise on public metric
    noise_priv: float = 0.01           # noise on private metric

    # welfare impact
    gamma_welfare: float = 0.7         # coefficient γ: how strongly gaming harms welfare

    # participation cost / decision parameters
    base_cost_mean: float = 0.4        # mean participation cost across clients
    base_cost_std: float = 0.1         # std dev of participation cost
    participation_beta: float = 4.0    # logistic slope (larger → sharper threshold)
    participation_bias: float = 0.0    # logistic offset (baseline profit threshold)

    seed: Optional[int] = None         # random seed (for reproducibility)


# ===================================================================
# 2. Client initialization
# ===================================================================

def init_clients(cfg: SimulationConfig) -> Dict[str, Any]:
    """
    For each client, initialize:
    - type: 'honest' or 'gaming'
    - cost: participation cost
    """
    rng = np.random.default_rng(cfg.seed)

    n = cfg.n_clients
    n_gaming = int(np.round(cfg.gaming_frac * n))

    client_type = np.array(['honest'] * n, dtype=object)
    if n_gaming > 0:
        gaming_idx = rng.choice(n, size=n_gaming, replace=False)
        client_type[gaming_idx] = 'gaming'

    # participation cost: controlled by mean/std in config
    cost = rng.normal(loc=cfg.base_cost_mean,
                      scale=cfg.base_cost_std,
                      size=n)
    cost = np.clip(cost, 0.05, None)  # avoid negative or too small costs

    return {
        "client_type": client_type,
        "cost": cost,
        "rng": rng,
    }


# ===================================================================
# 3. Single-round simulation
# ===================================================================

def simulate_round(
    cfg: SimulationConfig,
    state: Dict[str, Any],
    M_prev: float
) -> Dict[str, Any]:

    client_type = state["client_type"]
    cost = state["cost"]
    rng = state["rng"]

    n = cfg.n_clients

    # ---------------------------------------------------------------
    # 3.1 Participation decision: logistic-based probabilistic join
    # ---------------------------------------------------------------
    approx_p_audit = cfg.audit_budget / max(n, 1)

    expected_penalty_gaming = cfg.alpha_penalty * approx_p_audit * cfg.p_detect
    expected_penalty_honest = 0.0

    profit = np.zeros(n)

    mask_g = (client_type == 'gaming')
    mask_h = (client_type == 'honest')

    profit[mask_g] = (
        cfg.reward_bias
        + cfg.base_reward * M_prev
        - expected_penalty_gaming
        - cost[mask_g]
    )
    profit[mask_h] = (
        cfg.reward_bias
        + cfg.base_reward * M_prev
        - expected_penalty_honest
        - cost[mask_h]
    )

    logits = cfg.participation_beta * (profit - cfg.participation_bias)
    p_participate = sigmoid(logits)

    participate = rng.random(size=n) < p_participate
    participants_idx = np.where(participate)[0]
    n_participants = len(participants_idx)

    # ---------------------------------------------------------------
    # 3.2 Number of honest / gaming participants and participation rate
    # ---------------------------------------------------------------
    if n_participants > 0:
        types_participants = client_type[participants_idx]
        H = int(np.sum(types_participants == 'honest'))
        G = int(np.sum(types_participants == 'gaming'))
    else:
        H = 0
        G = 0

    x_t = n_participants / n

    # ---------------------------------------------------------------
    # 3.3 Welfare and metric computation
    # ---------------------------------------------------------------
    if n == 0:
        W_t = 0.0
    else:
        W_t = (H - cfg.gamma_welfare * G) / n
        W_t = float(np.clip(W_t, 0.0, 1.0))

    delta_metric = 0.3
    noise_pub = rng.normal(0.0, cfg.noise_pub)
    noise_priv = rng.normal(0.0, cfg.noise_priv)

    M_pub = W_t + delta_metric * (G / max(n, 1)) + noise_pub
    M_priv = W_t + noise_priv

    M_t = cfg.rho_pub * M_pub + (1.0 - cfg.rho_pub) * M_priv
    M_t = float(np.clip(M_t, 0.0, 1.0))

    # ---------------------------------------------------------------
    # 3.4 Auditing and penalties
    # ---------------------------------------------------------------
    audited_idx = np.array([], dtype=int)
    penalized_idx = np.array([], dtype=int)

    if n_participants > 0 and cfg.audit_budget > 0:
        k = min(cfg.audit_budget, n_participants)
        audited_idx = rng.choice(participants_idx, size=k, replace=False)

        is_gaming_audited = (client_type[audited_idx] == 'gaming')
        detect_mask = rng.random(size=k) < (cfg.p_detect * is_gaming_audited.astype(float))
        penalized_idx = audited_idx[detect_mask]

    rewards = np.zeros(n)
    if n_participants > 0:
        rewards[participants_idx] = cfg.base_reward * M_t - cost[participants_idx]
        rewards[penalized_idx] -= cfg.alpha_penalty

    return {
        "W_t": W_t,
        "M_t": M_t,
        "x_t": x_t,
        "H_t": H,
        "G_t": G,
        "n_participants": n_participants,
        "audited_idx": audited_idx,
        "penalized_idx": penalized_idx,
        "rewards": rewards,
    }


# ===================================================================
# 4. Full simulation runner
# ===================================================================

def run_simulation(cfg: SimulationConfig) -> pd.DataFrame:
    """
    Run T rounds of simulation under a single policy configuration (cfg)
    and return a DataFrame of per-round logs.
    Logged fields: W_t, M_t, x_t, H_t, G_t, n_participants, n_audited, n_penalized.
    """
    state = init_clients(cfg)
    M_prev = 0.6  # initial metric

    logs: List[Dict[str, Any]] = []

    for t in range(cfg.T):
        round_result = simulate_round(cfg, state, M_prev)

        logs.append({
            "t": t,
            "W_t": round_result["W_t"],
            "M_t": round_result["M_t"],
            "x_t": round_result["x_t"],
            "H_t": round_result["H_t"],
            "G_t": round_result["G_t"],
            "n_participants": round_result["n_participants"],
            "n_audited": len(round_result["audited_idx"]),
            "n_penalized": len(round_result["penalized_idx"]),
        })

        M_prev = round_result["M_t"]

    df = pd.DataFrame(logs)
    return df


# ===================================================================
# 5. Helper for PoG estimation (aligned vs gaming)
# ===================================================================

def estimate_price_of_gaming(
    cfg_aligned: SimulationConfig,
    cfg_gaming: SimulationConfig,
    burn_in: int = 100
) -> Dict[str, Any]:
    """
    Compare two scenarios to approximate PoG:
    - cfg_aligned: gaming_frac = 0
    - cfg_gaming: gaming_frac > 0

    We approximate steady state by averaging W_t, x_t, M_t after 'burn_in' rounds.
    """

    df_align = run_simulation(cfg_aligned)
    df_game = run_simulation(cfg_gaming)

    mask_align = (df_align["t"] >= burn_in)
    mask_game = (df_game["t"] >= burn_in)

    W_align = df_align.loc[mask_align, "W_t"].mean()
    W_game = df_game.loc[mask_game, "W_t"].mean()

    x_align = df_align.loc[mask_align, "x_t"].mean()
    x_game = df_game.loc[mask_game, "x_t"].mean()

    M_align = df_align.loc[mask_align, "M_t"].mean()
    M_game = df_game.loc[mask_game, "M_t"].mean()

    if W_align <= 0:
        PoG = np.nan
    else:
        PoG = (W_align - W_game) / W_align

    return {
        "W_align": W_align,
        "W_game": W_game,
        "x_align": x_align,
        "x_game": x_game,
        "M_align": M_align,
        "M_game": M_game,
        "PoG": PoG,
        "cfg_aligned": asdict(cfg_aligned),
        "cfg_gaming": asdict(cfg_gaming),
        "df_aligned": df_align,
        "df_gaming": df_game,
    }


# ===================================================================
# 6. Experiment 1: sweep alpha_penalty vs PoG
# ===================================================================

def sweep_alpha(
    base_cfg: SimulationConfig,
    alphas: List[float],
    burn_in: int = 100,
) -> pd.DataFrame:
    """
    Vary alpha_penalty and measure PoG and participation.
    """
    rows = []

    for alpha in alphas:
        cfg_gaming = SimulationConfig(**{**asdict(base_cfg),
                                         "gaming_frac": base_cfg.gaming_frac,
                                         "alpha_penalty": alpha})
        cfg_aligned = SimulationConfig(**{**asdict(base_cfg),
                                          "gaming_frac": 0.0,
                                          "alpha_penalty": alpha})

        result = estimate_price_of_gaming(cfg_aligned, cfg_gaming, burn_in=burn_in)

        rows.append({
            "alpha_penalty": alpha,
            "W_align": result["W_align"],
            "W_game": result["W_game"],
            "x_align": result["x_align"],
            "x_game": result["x_game"],
            "M_align": result["M_align"],
            "M_game": result["M_game"],
            "PoG": result["PoG"],
        })

    return pd.DataFrame(rows)


# ===================================================================
# 7. Experiment 2: sweep rho_pub (public metric weight) vs PoG
# ===================================================================

def sweep_rho_pub(
    base_cfg: SimulationConfig,
    rhos: List[float],
    burn_in: int = 100,
) -> pd.DataFrame:
    """
    Vary rho_pub (weight on public metric) and measure PoG and M-W gap.
    """
    rows = []

    for rho in rhos:
        cfg_gaming = SimulationConfig(**{**asdict(base_cfg),
                                         "gaming_frac": base_cfg.gaming_frac,
                                         "rho_pub": rho})
        cfg_aligned = SimulationConfig(**{**asdict(base_cfg),
                                          "gaming_frac": 0.0,
                                          "rho_pub": rho})

        result = estimate_price_of_gaming(cfg_aligned, cfg_gaming, burn_in=burn_in)

        # M - W gap (metric inflation) under each scenario
        M_W_gap_align = result["M_align"] - result["W_align"]
        M_W_gap_game = result["M_game"] - result["W_game"]

        rows.append({
            "rho_pub": rho,
            "W_align": result["W_align"],
            "W_game": result["W_game"],
            "x_align": result["x_align"],
            "x_game": result["x_game"],
            "M_align": result["M_align"],
            "M_game": result["M_game"],
            "M_W_gap_align": M_W_gap_align,
            "M_W_gap_game": M_W_gap_game,
            "PoG": result["PoG"],
        })

    return pd.DataFrame(rows)


# ===================================================================
# 8. Experiment 3: participation vs alpha (resilience / stability)
# ===================================================================

def sweep_alpha_participation(
    base_cfg: SimulationConfig,
    alphas: List[float],
    burn_in: int = 100,
) -> pd.DataFrame:
    """
    Vary alpha_penalty and, under the gaming scenario, measure average participation (x_t).
    (We only track the gaming case here rather than aligned vs gaming.)
    """
    rows = []

    for alpha in alphas:
        cfg_gaming = SimulationConfig(**{**asdict(base_cfg),
                                         "alpha_penalty": alpha})

        df_game = run_simulation(cfg_gaming)
        mask_game = (df_game["t"] >= burn_in)

        x_mean = df_game.loc[mask_game, "x_t"].mean()
        W_mean = df_game.loc[mask_game, "W_t"].mean()

        rows.append({
            "alpha_penalty": alpha,
            "x_game_mean": x_mean,
            "W_game_mean": W_mean,
        })

    return pd.DataFrame(rows)


# ===================================================================
# 9. Simple usage example (only when run as a script)
# ===================================================================

if __name__ == "__main__":
    # Base configuration (near the "bad policy" region used in the discussion)
    base_cfg = SimulationConfig(
        n_clients=100,
        T=300,
        gaming_frac=0.3,
        base_reward=1.0,
        reward_bias=0.2,
        audit_budget=10,
        alpha_penalty=0.7,
        p_detect=0.7,
        rho_pub=0.6,
        noise_pub=0.01,
        noise_priv=0.01,
        gamma_welfare=0.7,
        base_cost_mean=0.4,
        base_cost_std=0.1,
        participation_beta=4.0,
        participation_bias=0.0,
        seed=42,
    )

    print("=== Single simulation example (gaming scenario) ===")
    df_example = run_simulation(base_cfg)
    print(df_example.head())
    print(df_example.tail())

    print("\n=== PoG (aligned vs gaming) example ===")
    cfg_aligned = SimulationConfig(**{**asdict(base_cfg), "gaming_frac": 0.0})
    result = estimate_price_of_gaming(cfg_aligned, base_cfg, burn_in=100)
    print(f"W_align ≈ {result['W_align']:.3f}")
    print(f"W_game  ≈ {result['W_game']:.3f}")
    print(f"x_align ≈ {result['x_align']:.3f}")
    print(f"x_game  ≈ {result['x_game']:.3f}")
    print(f"PoG     ≈ {result['PoG']:.3f}")

    print("\n=== Experiment 1: PoG vs alpha_penalty ===")
    alphas = [0.3, 0.5, 0.7, 1.0, 1.5]
    df_alpha = sweep_alpha(base_cfg, alphas, burn_in=100)
    print(df_alpha)

    print("\n=== Experiment 2: PoG vs rho_pub ===")
    rhos = [1.0, 0.8, 0.6, 0.4, 0.2]
    df_rho = sweep_rho_pub(base_cfg, rhos, burn_in=100)
    print(df_rho)

    print("\n=== Experiment 3: participation vs alpha (gaming only) ===")
    df_alpha_part = sweep_alpha_participation(base_cfg, alphas, burn_in=100)
    print(df_alpha_part)

# Real-World Federated Learning

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms

# ============================================================
# 0. Config / Seed
# ============================================================

class Config:
    n_clients = 30
    gaming_frac = 0.3
    n_rounds = 40
    local_epochs = 2
    batch_size = 64
    lr = 0.01
    momentum = 0.9

    # Dirichlet non-IID partition parameter
    dirichlet_alpha = 0.5

    # Fraction of public validation data leaked to gaming clients
    leak_fraction = 1.0  # use entire head-only public val for maximum effect

    # Class split
    head_classes = {0, 1, 2, 3, 4}   # head classes used by the public metric
    tail_classes = {5, 6, 7, 8, 9}   # tail classes used by welfare

    seed = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(Config.seed)

# ============================================================
# 1. Dataset preparation (Fashion-MNIST)
#    - train 60k -> train_local 50k + public_val (head-biased) ~10k
#    - test 10k  -> hidden welfare evaluation on tail-only
# ============================================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

root = "./data"

full_train = datasets.FashionMNIST(root=root, train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(root=root, train=False, download=True, transform=transform)

# From the 60k training examples:
#   first 50k: local training
#   last 10k : public validation candidates
train_local_size = 50_000
indices_all = np.arange(len(full_train))
train_local_indices = indices_all[:train_local_size]
public_val_candidate_indices = indices_all[train_local_size:]

train_local_set = Subset(full_train, train_local_indices)

# Public validation: only head classes
all_train_targets = np.array(full_train.targets)
head_mask = np.isin(all_train_targets, list(Config.head_classes))
public_val_head_indices = [idx for idx in public_val_candidate_indices if head_mask[idx]]

public_val_head_indices = np.array(public_val_head_indices)
rng = np.random.default_rng(Config.seed)
rng.shuffle(public_val_head_indices)

leak_size = int(Config.leak_fraction * len(public_val_head_indices))
leak_indices = public_val_head_indices[:leak_size]

public_val_set = Subset(full_train, public_val_head_indices)
public_val_loader = DataLoader(public_val_set, batch_size=Config.batch_size, shuffle=False)

# Hidden welfare: only tail classes from the test set
all_test_targets = np.array(test_set.targets)
tail_mask = np.isin(all_test_targets, list(Config.tail_classes))
tail_test_indices = np.where(tail_mask)[0]
hidden_tail_test_set = Subset(test_set, tail_test_indices)
hidden_tail_test_loader = DataLoader(hidden_tail_test_set, batch_size=Config.batch_size, shuffle=False)

# (Optional) If we want full test accuracy later
full_test_loader = DataLoader(test_set, batch_size=Config.batch_size, shuffle=False)

# Public leak subset for gaming clients (head-only)
leak_subset = Subset(full_train, leak_indices)


# ============================================================
# 2. Non-IID partitioning (Dirichlet) for client local datasets
# ============================================================

def make_dirichlet_partitions(labels: np.ndarray, n_clients: int, alpha: float, rng: np.random.Generator):
    """
    labels: 1D array of class labels for train_local_set
    n_clients: number of clients
    alpha: Dirichlet concentration parameter

    Returns:
        list of index arrays per client (indices are with respect to train_local_set)
    """
    n_classes = int(labels.max()) + 1
    client_indices = [[] for _ in range(n_clients)]

    # Collect indices per class (indices are within train_local_set)
    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue

        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shard = np.split(idx_k, splits[:-1])

        for cid, shard_c in enumerate(shard):
            client_indices[cid].extend(shard_c.tolist())

    for cid in range(n_clients):
        rng.shuffle(client_indices[cid])

    return client_indices


rng = np.random.default_rng(Config.seed)

# Extract labels for train_local_set (in the order of train_local_indices)
train_targets = np.array(full_train.targets)[train_local_indices]

client_indices = make_dirichlet_partitions(
    labels=train_targets,
    n_clients=Config.n_clients,
    alpha=Config.dirichlet_alpha,
    rng=rng
)

client_datasets_base = [
    Subset(train_local_set, idxs) for idxs in client_indices
]


# ============================================================
# 3. Helper to filter a Subset by allowed labels
# ============================================================

def filter_subset_by_labels(base_subset: Subset, allowed_labels: set[int]) -> Subset:
    """
    base_subset: Subset(train_local_set, ...)
    allowed_labels: set of class labels to keep (e.g., head_classes)

    Returns:
        A new Subset(train_local_set, ...) that only includes samples whose
        labels are in allowed_labels.
    """
    assert isinstance(base_subset.dataset, Subset) or isinstance(base_subset.dataset, datasets.FashionMNIST) \
        or isinstance(base_subset.dataset, torch.utils.data.Dataset)

    # base_subset is a subset of train_local_set.
    # train_local_set.indices: indices into full_train.
    # base_subset.indices: indices within train_local_set (0~49999).
    kept_indices_in_train_local = []

    for idx_in_train_local in base_subset.indices:
        global_idx = train_local_set.indices[idx_in_train_local]
        label = int(full_train.targets[global_idx])
        if label in allowed_labels:
            kept_indices_in_train_local.append(idx_in_train_local)

    return Subset(train_local_set, kept_indices_in_train_local)


# ============================================================
# 4. CNN model definition
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 1x28x28 -> 32x28x28
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             # 32x14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 64x14x14
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),                             # 64x7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def get_model():
    model = SimpleCNN(num_classes=10)
    return model.to(Config.device)


# ============================================================
# 5. Evaluation function (accuracy)
# ============================================================

def evaluate_model(model: nn.Module, data_loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(Config.device)
            y = y.to(Config.device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total if total > 0 else 0.0


# ============================================================
# 6. Local training
#    (honest vs gaming is controlled by which dataset they receive)
# ============================================================

def local_train(model: nn.Module, dataset, epochs: int) -> nn.Module:
    loader = DataLoader(dataset, batch_size=Config.batch_size, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=Config.lr, momentum=Config.momentum)

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x = x.to(Config.device)
            y = y.to(Config.device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
    return model


# ============================================================
# 7. FedAvg aggregation
# ============================================================

def fedavg(models: list[nn.Module]) -> nn.Module:
    global_model = get_model()
    global_state = global_model.state_dict()

    state_dicts = [m.state_dict() for m in models]

    with torch.no_grad():
        for key in global_state.keys():
            stacked = torch.stack([sd[key] for sd in state_dicts], dim=0)
            global_state[key] = stacked.mean(dim=0)

    global_model.load_state_dict(global_state)
    return global_model


# ============================================================
# 8. Federated learning experiment (only gaming_frac changes)
# ============================================================

def run_federated_experiment(gaming_frac: float):
    """
    gaming_frac: 0.0 => all honest (aligned)
                 0.3 => 30% gaming

    Returns:
      - W_history: per-round hidden tail test accuracy (welfare)
      - M_history: per-round head-only public validation accuracy (metric)
      - overall_test_history: per-round full test accuracy (for reference)
    """
    n_clients = Config.n_clients
    n_gaming = int(round(gaming_frac * n_clients))

    all_client_ids = np.arange(n_clients)
    rng_local = np.random.default_rng(Config.seed)  # fix seed to keep client composition consistent
    gaming_clients = set(rng_local.choice(all_client_ids, size=n_gaming, replace=False))
    honest_clients = [cid for cid in all_client_ids if cid not in gaming_clients]

    print(f"[Run] gaming_frac = {gaming_frac:.2f}, "
          f"honest = {len(honest_clients)}, gaming = {len(gaming_clients)}")

    # Construct training dataset for each client
    client_train_datasets = []
    for cid in range(n_clients):
        base_ds = client_datasets_base[cid]
        if cid in gaming_clients and gaming_frac > 0:
            # Gaming client:
            #   1) Remove tail classes from local data (head-only)
            #   2) Concatenate head-only public leak
            base_head_only = filter_subset_by_labels(base_ds, Config.head_classes)
            ds = ConcatDataset([base_head_only, leak_subset])
        else:
            # Honest client: use entire local data (head + tail)
            ds = base_ds
        client_train_datasets.append(ds)

    global_model = get_model()

    W_history = []
    M_history = []
    overall_test_history = []

    for rnd in range(Config.n_rounds):
        print(f"=== Round {rnd+1}/{Config.n_rounds} ===")

        local_models = []
        for cid in range(n_clients):
            lm = get_model()
            lm.load_state_dict(global_model.state_dict())
            lm = local_train(lm, client_train_datasets[cid], epochs=Config.local_epochs)
            local_models.append(lm)

        global_model = fedavg(local_models)

        # Evaluation:
        # - W_t: tail-only hidden test (welfare)
        # - M_t: head-only public validation (metric)
        # - overall test accuracy as an additional reference
        W_t = evaluate_model(global_model, hidden_tail_test_loader)
        M_t = evaluate_model(global_model, public_val_loader)
        overall_acc = evaluate_model(global_model, full_test_loader)

        W_history.append(W_t)
        M_history.append(M_t)
        overall_test_history.append(overall_acc)

        print(f"  Hidden tail test acc (W_t) : {W_t:.4f}")
        print(f"  Public head val acc (M_t)  : {M_t:.4f}")
        print(f"  Full test acc (ref)       : {overall_acc:.4f}")

    return W_history, M_history, overall_test_history


# ============================================================
# 9. Run aligned vs gaming experiments and summarize
# ============================================================

def tail_mean(vals, tail=10):
    vals = np.array(vals)
    if len(vals) < tail:
        return vals.mean()
    return vals[-tail:].mean()


if __name__ == "__main__":
    # aligned (no gaming)
    W_aligned, M_aligned, full_aligned = run_federated_experiment(gaming_frac=0.0)

    # gaming (30% gaming)
    W_gaming, M_gaming, full_gaming = run_federated_experiment(gaming_frac=Config.gaming_frac)

    W_align_mean = tail_mean(W_aligned)
    W_game_mean = tail_mean(W_gaming)
    M_align_mean = tail_mean(M_aligned)
    M_game_mean = tail_mean(M_gaming)
    full_align_mean = tail_mean(full_aligned)
    full_game_mean = tail_mean(full_gaming)

    if W_align_mean > 0:
        PoG = (W_align_mean - W_game_mean) / W_align_mean
    else:
        PoG = float("nan")

    print("\n=== Summary (last 10 rounds, head-metric vs tail-welfare) ===")
    print(f"W_align (tail) ≈ {W_align_mean:.3f}")
    print(f"W_game  (tail) ≈ {W_game_mean:.3f}")
    print(f"M_align (head) ≈ {M_align_mean:.3f}")
    print(f"M_game  (head) ≈ {M_game_mean:.3f}")
    print(f"Full test (aligned) ≈ {full_align_mean:.3f}")
    print(f"Full test (gaming)  ≈ {full_game_mean:.3f}")
    print(f"PoG (tail welfare)  ≈ {PoG:.3f}")

# Estimator reliability under partial audits

In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms


# ============================================================
# E1: Estimator Reliability (Ground-truth vs Audit-based estimator)
# Standalone code (not meant to be merged into your old script)
# ============================================================

# ----------------------------
# 0) Config / Seed
# ----------------------------
@dataclass
class Config:
    # Keep small for Colab
    n_clients: int = 12
    n_rounds: int = 25
    local_epochs: int = 1
    batch_size: int = 64
    lr: float = 0.01
    momentum: float = 0.9

    # Non-IID partition
    dirichlet_alpha: float = 0.5

    # Head/Tail split
    head_classes: Tuple[int, ...] = (0, 1, 2, 3, 4)
    tail_classes: Tuple[int, ...] = (5, 6, 7, 8, 9)

    # Public leak
    leak_fraction: float = 1.0  # use all head-only public val for max gaming effect

    # Strategy mix for profiles
    gaming_fracs: Tuple[float, ...] = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5)     # for GT / estimator comparison
    benign_frac: float = 0.10                               # small benign cooperation fraction
    update_scale_factor: float = 1.75                       # for update-scaling gaming

    # Audit budgets to test estimator reliability
    audit_budgets: Tuple[float, ...] = (0.10, 0.25, 0.50)
    audit_trials: int = 5  # resample audits to estimate estimator noise

    # Evaluation
    tail_k: int = 7          # tail-mean over last K rounds
    pog_threshold: float = 0.05  # risk threshold for FP/FN

    seed: int = 42
    root: str = "./data"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG = Config()


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)


# ----------------------------
# 1) Dataset preparation
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = datasets.FashionMNIST(root=CFG.root, train=True, download=True, transform=transform)

# We'll use:
# - 50k for client-local data (partitioned)
# - 10k as public validation candidates (from the same training set)
train_local_size = 50_000
idx_all = np.arange(len(full_train))
idx_local = idx_all[:train_local_size]
idx_public_candidates = idx_all[train_local_size:]

train_local_set = Subset(full_train, idx_local)

targets_all = np.array(full_train.targets)

# Public validation set: head-only from last 10k
head_mask = np.isin(targets_all, list(CFG.head_classes))
public_head_indices = [i for i in idx_public_candidates if head_mask[i]]
public_head_indices = np.array(public_head_indices)

rng = np.random.default_rng(CFG.seed)
rng.shuffle(public_head_indices)

leak_size = int(CFG.leak_fraction * len(public_head_indices))
leak_indices = public_head_indices[:leak_size]

public_val_set = Subset(full_train, public_head_indices)
public_val_loader = DataLoader(public_val_set, batch_size=CFG.batch_size, shuffle=False)

# Leak subset that gaming clients can append
leak_subset = Subset(full_train, leak_indices)


# ----------------------------
# 2) Utilities: partition + filtering
# ----------------------------
def make_dirichlet_partitions(labels: np.ndarray, n_clients: int, alpha: float, rng: np.random.Generator):
    """Returns list of index arrays per client (indices w.r.t. train_local_set)."""
    n_classes = int(labels.max()) + 1
    client_indices: List[List[int]] = [[] for _ in range(n_clients)]

    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue
        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shards = np.split(idx_k, splits[:-1])
        for cid, shard in enumerate(shards):
            client_indices[cid].extend(shard.tolist())

    for cid in range(n_clients):
        rng.shuffle(client_indices[cid])

    return client_indices


def filter_subset_by_labels(base_subset: Subset, allowed_labels: Tuple[int, ...]) -> Subset:
    """base_subset is a Subset of train_local_set; keep only items with labels in allowed_labels."""
    allowed = set(allowed_labels)
    kept = []
    # base_subset.indices are indices inside train_local_set (0..train_local_size-1)
    for idx_in_local in base_subset.indices:
        global_idx = train_local_set.indices[idx_in_local]  # index into full_train
        y = int(full_train.targets[global_idx])
        if y in allowed:
            kept.append(idx_in_local)
    return Subset(train_local_set, kept)


def split_subset_train_eval(base_subset: Subset, eval_ratio: float = 0.2) -> Tuple[Subset, Subset]:
    """Split a client's local subset into train/eval (by index)."""
    idxs = list(base_subset.indices)
    rng_local = np.random.default_rng(CFG.seed + 123)
    rng_local.shuffle(idxs)
    n_eval = int(round(eval_ratio * len(idxs)))
    eval_idxs = idxs[:n_eval]
    train_idxs = idxs[n_eval:]
    return Subset(train_local_set, train_idxs), Subset(train_local_set, eval_idxs)


def tail_only_subset(base_subset: Subset) -> Subset:
    return filter_subset_by_labels(base_subset, CFG.tail_classes)


def oversample_tail(train_subset: Subset, factor: int = 2) -> ConcatDataset:
    """Benign cooperation: upweight tail samples in local training."""
    tail_sub = tail_only_subset(train_subset)
    # If tail is empty, just return original
    if len(tail_sub) == 0:
        return ConcatDataset([train_subset])
    reps = [tail_sub for _ in range(factor)]
    return ConcatDataset([train_subset] + reps)


# Build base client datasets (Dirichlet)
train_targets_local = np.array(full_train.targets)[idx_local]
client_indices = make_dirichlet_partitions(
    labels=train_targets_local,
    n_clients=CFG.n_clients,
    alpha=CFG.dirichlet_alpha,
    rng=np.random.default_rng(CFG.seed)
)

client_base_subsets = [Subset(train_local_set, idxs) for idxs in client_indices]

# For auditing/welfare evaluation, each client has a held-out local eval set; welfare = tail-accuracy on that eval set.
client_train_subsets: List[Subset] = []
client_eval_subsets: List[Subset] = []
client_tail_eval_subsets: List[Subset] = []

for cid in range(CFG.n_clients):
    tr, ev = split_subset_train_eval(client_base_subsets[cid], eval_ratio=0.2)
    client_train_subsets.append(tr)
    client_eval_subsets.append(ev)
    client_tail_eval_subsets.append(tail_only_subset(ev))


# ----------------------------
# 3) Model
# ----------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def get_model() -> nn.Module:
    return SimpleCNN().to(CFG.device)


# ----------------------------
# 4) Train/Eval helpers
# ----------------------------
@torch.no_grad()
def evaluate_acc(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(CFG.device), y.to(CFG.device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


def local_train(model: nn.Module, dataset, epochs: int) -> nn.Module:
    loader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=True)
    opt = optim.SGD(model.parameters(), lr=CFG.lr, momentum=CFG.momentum)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(CFG.device), y.to(CFG.device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
    return model


def state_dict_add_scaled_delta(global_sd: Dict[str, torch.Tensor],
                                local_sd: Dict[str, torch.Tensor],
                                scale: float) -> Dict[str, torch.Tensor]:
    """Return transmitted state = global + scale*(local - global)."""
    out = {}
    for k in global_sd.keys():
        out[k] = global_sd[k] + scale * (local_sd[k] - global_sd[k])
    return out


def fedavg_from_states(states: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    """Average a list of state_dicts (FedAvg)."""
    avg = {}
    for k in states[0].keys():
        stacked = torch.stack([sd[k] for sd in states], dim=0)
        avg[k] = stacked.mean(dim=0)
    return avg


# ----------------------------
# 5) Strategy definitions
# ----------------------------
STR_HONEST = "HONEST"
STR_BENIGN = "BENIGN_TAIL_UPWEIGHT"
STR_GAME_HEAD = "GAMING_HEAD_ONLY_LEAK"
STR_GAME_SCALE = "GAMING_UPDATE_SCALING"


def assign_strategies(n_clients: int,
                      gaming_frac: float,
                      benign_frac: float,
                      rng: np.random.Generator) -> Dict[int, str]:
    """
    Assign each client a strategy from {HONEST, BENIGN, GAMING_HEAD, GAMING_SCALE}.
    To keep it simple:
      - benign_frac portion get BENIGN
      - gaming_frac portion get a GAMING strategy (half head-only, half update-scaling)
      - rest are HONEST
    """
    all_ids = np.arange(n_clients)
    rng.shuffle(all_ids)

    n_benign = int(round(benign_frac * n_clients))
    n_gaming = int(round(gaming_frac * n_clients))

    benign_ids = set(all_ids[:n_benign])
    gaming_ids = list(all_ids[n_benign:n_benign + n_gaming])
    honest_ids = set(all_ids[n_benign + n_gaming:])

    strat = {}
    for cid in benign_ids:
        strat[cid] = STR_BENIGN
    for cid in honest_ids:
        strat[cid] = STR_HONEST

    # split gaming IDs into two types
    half = len(gaming_ids) // 2
    for cid in gaming_ids[:half]:
        strat[cid] = STR_GAME_HEAD
    for cid in gaming_ids[half:]:
        strat[cid] = STR_GAME_SCALE

    return strat


def build_client_train_dataset(cid: int, strategy: str):
    base_train = client_train_subsets[cid]

    if strategy == STR_HONEST:
        return base_train

    if strategy == STR_BENIGN:
        # upweight tail a bit (benign cooperation)
        return oversample_tail(base_train, factor=2)

    if strategy == STR_GAME_HEAD:
        # Remove tail samples from local training; append public head leak
        head_only = filter_subset_by_labels(base_train, CFG.head_classes)
        return ConcatDataset([head_only, leak_subset])

    if strategy == STR_GAME_SCALE:
        # Training data itself is honest; manipulation happens in transmitted update
        return base_train

    raise ValueError(f"Unknown strategy: {strategy}")


# ----------------------------
# 6) Run FL once per profile, store per-round:
#    - public metric M_t (head-only public val accuracy)
#    - per-client tail accuracies on client-heldout tail eval (for GT + audit sampling)
# ----------------------------
def run_profile_once(gaming_frac: float, benign_frac: float, seed_offset: int = 0):
    rng = np.random.default_rng(CFG.seed + seed_offset)

    strat_map = assign_strategies(CFG.n_clients, gaming_frac, benign_frac, rng)

    # Pre-build each client's training dataset according to strategy
    client_train_ds = [build_client_train_dataset(cid, strat_map[cid]) for cid in range(CFG.n_clients)]

    global_model = get_model()
    global_sd = {k: v.detach().clone() for k, v in global_model.state_dict().items()}

    M_hist: List[float] = []
    tail_acc_matrix = np.zeros((CFG.n_rounds, CFG.n_clients), dtype=np.float32)

    for rnd in range(CFG.n_rounds):
        transmitted_states: List[Dict[str, torch.Tensor]] = []

        for cid in range(CFG.n_clients):
            local_model = get_model()
            local_model.load_state_dict(global_sd)

            local_model = local_train(local_model, client_train_ds[cid], epochs=CFG.local_epochs)
            local_sd = local_model.state_dict()

            # Manipulation at transmission time for UPDATE_SCALING gaming
            if strat_map[cid] == STR_GAME_SCALE and gaming_frac > 0:
                tx_sd = state_dict_add_scaled_delta(global_sd, local_sd, scale=CFG.update_scale_factor)
            else:
                tx_sd = {k: v.detach().clone() for k, v in local_sd.items()}

            transmitted_states.append(tx_sd)

        # FedAvg update
        global_sd = fedavg_from_states(transmitted_states)
        global_model.load_state_dict(global_sd)

        # Metric: server-observable public head validation accuracy
        M_t = evaluate_acc(global_model, public_val_loader)
        M_hist.append(M_t)

        # Welfare ingredient: per-client tail eval accuracy (experimenter can compute for GT;
        # audit will subsample from this vector)
        for cid in range(CFG.n_clients):
            tail_ev = client_tail_eval_subsets[cid]
            if len(tail_ev) == 0:
                tail_acc_matrix[rnd, cid] = np.nan  # if a client has no tail samples
                continue
            loader = DataLoader(tail_ev, batch_size=CFG.batch_size, shuffle=False)
            tail_acc_matrix[rnd, cid] = evaluate_acc(global_model, loader)

        if (rnd + 1) % 10 == 0 or rnd == 0:
            w_full = np.nanmean(tail_acc_matrix[rnd])
            print(f"[Profile gf={gaming_frac:.2f}] Round {rnd+1:>2}/{CFG.n_rounds} | "
                  f"M(head public)={M_t:.4f} | W_full_tail(mean clients)={w_full:.4f}")

    return {
        "gaming_frac": gaming_frac,
        "benign_frac": benign_frac,
        "strategy_map": strat_map,
        "M_hist": np.array(M_hist, dtype=np.float32),
        "tail_acc_matrix": tail_acc_matrix,  # shape [T, N]
    }


def tail_mean(x: np.ndarray, k: int) -> float:
    if len(x) == 0:
        return float("nan")
    if len(x) <= k:
        return float(np.nanmean(x))
    return float(np.nanmean(x[-k:]))


# ----------------------------
# 7) Estimator: audit-based welfare estimate from tail acc matrix
#    - GT welfare per round: mean over all clients (nanmean)
#    - Estimated welfare per round: mean over audited clients (nanmean over sampled subset)
#    - PoG uses baseline welfare from aligned profile
# ----------------------------
def compute_pog_series(w_ref: float, w_series: np.ndarray) -> np.ndarray:
    # PoG_t = (W_ref - W_t) / W_ref
    if not np.isfinite(w_ref) or w_ref <= 1e-8:
        return np.full_like(w_series, np.nan, dtype=np.float32)
    out = (w_ref - w_series) / w_ref
    return out.astype(np.float32)


def audit_estimate_welfare_series(tail_acc_matrix: np.ndarray,
                                 audit_budget: float,
                                 rng: np.random.Generator) -> np.ndarray:
    """
    tail_acc_matrix: [T, N] per-client tail acc (nan if no tail data)
    returns: W_hat_series [T]
    """
    T, N = tail_acc_matrix.shape
    m = max(1, int(round(audit_budget * N)))
    W_hat = np.zeros(T, dtype=np.float32)
    for t in range(T):
        audited = rng.choice(np.arange(N), size=m, replace=False)
        W_hat[t] = float(np.nanmean(tail_acc_matrix[t, audited]))
    return W_hat


def detection_delay(pog_hat: np.ndarray, tau: float) -> int:
    """First round index (1-based) where PoG_hat >= tau; return 0 if never detected."""
    for i, v in enumerate(pog_hat):
        if np.isfinite(v) and v >= tau:
            return i + 1
    return 0


# ----------------------------
# 8) Main: run baseline + profiles, then evaluate estimator reliability
# ----------------------------
def main():
    print("\n=== E1: Estimator Reliability (GT vs Audit-based Estimator) ===")
    print(f"Device: {CFG.device}")
    print(f"Clients={CFG.n_clients}, Rounds={CFG.n_rounds}, LocalEpochs={CFG.local_epochs}")
    print(f"Audit budgets={CFG.audit_budgets}, trials={CFG.audit_trials}\n")

    # 8.1 Baseline aligned run (gaming_frac=0.0), used as welfare reference
    baseline = run_profile_once(gaming_frac=0.0, benign_frac=CFG.benign_frac, seed_offset=0)
    M_ref_series = baseline["M_hist"]
    W_ref_series_full = np.nanmean(baseline["tail_acc_matrix"], axis=1)  # GT welfare series
    W_ref = tail_mean(W_ref_series_full, CFG.tail_k)
    M_ref = tail_mean(M_ref_series, CFG.tail_k)

    print("\n[Baseline] Reference values (tail-mean over last K rounds)")
    print(f"  W_ref (tail welfare) = {W_ref:.4f}")
    print(f"  M_ref (head metric)  = {M_ref:.4f}\n")

    # 8.2 Run profiles for each gaming_frac (one run each), store GT
    profiles = []
    seed_offset = 10
    for gf in CFG.gaming_fracs:
        prof = run_profile_once(gaming_frac=gf, benign_frac=CFG.benign_frac, seed_offset=seed_offset)
        seed_offset += 1
        profiles.append(prof)

    # 8.3 For each profile, compute GT PoG and estimator performance for each audit budget
    rows = []
    for prof in profiles:
        gf = prof["gaming_frac"]
        M_series = prof["M_hist"]
        tail_mat = prof["tail_acc_matrix"]

        # Ground-truth welfare series (full visibility, experimenter only)
        W_full_series = np.nanmean(tail_mat, axis=1)
        W_full = tail_mean(W_full_series, CFG.tail_k)

        # Ground-truth PoG (final)
        pog_gt_series = compute_pog_series(W_ref, W_full_series)
        pog_gt = tail_mean(pog_gt_series, CFG.tail_k)

        # Simple manipulation index proxy: metric inflation vs baseline (final)
        M_final = tail_mean(M_series, CFG.tail_k)
        manip_delta = M_final - M_ref

        # Risk label for FP/FN (based on GT)
        is_risky_gt = (np.isfinite(pog_gt) and pog_gt >= CFG.pog_threshold)

        print(f"\n[Profile gf={gf:.2f}] GT summary:")
        print(f"  M_final(head metric)  = {M_final:.4f} (delta vs ref {manip_delta:+.4f})")
        print(f"  W_full(tail welfare)  = {W_full:.4f}")
        print(f"  PoG_GT                = {pog_gt:.4f} (risk>=tau? {is_risky_gt})")

        # Estimator: for each budget, resample audits multiple times
        for b in CFG.audit_budgets:
            pog_hats = []
            delays = []
            risky_preds = []

            for tr in range(CFG.audit_trials):
                rng_a = np.random.default_rng(CFG.seed + 1000 + tr + int(100 * b) + int(1000 * gf))
                W_hat_series = audit_estimate_welfare_series(tail_mat, audit_budget=b, rng=rng_a)
                pog_hat_series = compute_pog_series(W_ref, W_hat_series)
                pog_hat = tail_mean(pog_hat_series, CFG.tail_k)

                pog_hats.append(pog_hat)
                delays.append(detection_delay(pog_hat_series, CFG.pog_threshold))
                risky_preds.append(np.isfinite(pog_hat) and pog_hat >= CFG.pog_threshold)

            pog_hat_mean = float(np.nanmean(pog_hats))
            pog_hat_std = float(np.nanstd(pog_hats))
            delay_mean = float(np.mean(delays))

            # Classification outcomes
            pred_risky = (np.mean(risky_preds) >= 0.5)  # majority vote across trials
            fp = int((not is_risky_gt) and pred_risky)
            fn = int(is_risky_gt and (not pred_risky))

            rows.append({
                "gaming_frac": gf,
                "audit_budget": b,
                "M_final": M_final,
                "M_delta_vs_ref": manip_delta,
                "W_full": W_full,
                "PoG_GT": pog_gt,
                "PoG_hat_mean": pog_hat_mean,
                "PoG_hat_std": pog_hat_std,
                "delay_mean_rounds": delay_mean,
                "risk_GT": int(is_risky_gt),
                "risk_pred_majority": int(pred_risky),
                "FP": fp,
                "FN": fn,
            })

            print(f"  [Audit b={b:.2f}] PoG_hat={pog_hat_mean:.4f}±{pog_hat_std:.4f} | "
                  f"delay~{delay_mean:.1f} rounds | pred_risky={pred_risky} | FP={fp} FN={fn}")

    # 8.4 Aggregate reliability metrics across profiles for each budget
    print("\n=== Aggregated Estimator Reliability (across profiles) ===")
    rows_by_b = {}
    for r in rows:
        rows_by_b.setdefault(r["audit_budget"], []).append(r)

    for b, rs in rows_by_b.items():
        gt = np.array([x["PoG_GT"] for x in rs], dtype=np.float32)
        hat = np.array([x["PoG_hat_mean"] for x in rs], dtype=np.float32)

        # Spearman rank correlation (manual, simple)
        def rankdata(a):
            temp = a.argsort()
            ranks = np.empty_like(temp)
            ranks[temp] = np.arange(len(a))
            return ranks.astype(np.float32)

        mask = np.isfinite(gt) & np.isfinite(hat)
        if mask.sum() >= 2:
            rg = rankdata(gt[mask])
            rh = rankdata(hat[mask])
            spearman = float(np.corrcoef(rg, rh)[0, 1])
        else:
            spearman = float("nan")

        FP = sum(x["FP"] for x in rs)
        FN = sum(x["FN"] for x in rs)
        n = len(rs)

        print(f"Audit b={b:.2f} | Spearman(PoG_GT, PoG_hat)={spearman:.3f} | "
              f"FP={FP}/{n} FN={FN}/{n}")

    # 8.5 Save CSV
    out_path = "E1_estimator_reliability_results.csv"
    import csv
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"\nSaved: {out_path}")
    print("Done.")


if __name__ == "__main__":
    main()


=== E1: Estimator Reliability (GT vs Audit-based Estimator) ===
Device: cuda
Clients=12, Rounds=25, LocalEpochs=1
Audit budgets=(0.1, 0.25, 0.5), trials=5

[Profile gf=0.00] Round  1/25 | M(head public)=0.6346 | W_full_tail(mean clients)=0.5981
[Profile gf=0.00] Round 10/25 | M(head public)=0.8438 | W_full_tail(mean clients)=0.8482
[Profile gf=0.00] Round 20/25 | M(head public)=0.8705 | W_full_tail(mean clients)=0.8552

[Baseline] Reference values (tail-mean over last K rounds)
  W_ref (tail welfare) = 0.8822
  M_ref (head metric)  = 0.8621

[Profile gf=0.00] Round  1/25 | M(head public)=0.5285 | W_full_tail(mean clients)=0.6210
[Profile gf=0.00] Round 10/25 | M(head public)=0.8358 | W_full_tail(mean clients)=0.8682
[Profile gf=0.00] Round 20/25 | M(head public)=0.8625 | W_full_tail(mean clients)=0.8922
[Profile gf=0.10] Round  1/25 | M(head public)=0.5904 | W_full_tail(mean clients)=0.6928
[Profile gf=0.10] Round 10/25 | M(head public)=0.8495 | W_full_tail(mean clients)=0.8594
[Profi

# Noise (privacy) and auditability

In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms


# ============================================================
# E2: Noise / Privacy Trade-off (DP-like update noise injection)
# Standalone code (separate from your E1 script)
#
# Changes requested:
# - noise_multipliers = (0.00, 0.05, 0.10)
# - keep the rest of settings the same as the previous E2 code
# - add more measured metrics
# ============================================================

# ----------------------------
# 0) Config / Seed
# ----------------------------
@dataclass
class Config:
    # Keep small for Colab
    n_clients: int = 12
    n_rounds: int = 25
    local_epochs: int = 1
    batch_size: int = 64
    lr: float = 0.01
    momentum: float = 0.9

    # Non-IID partition
    dirichlet_alpha: float = 0.5

    # Head/Tail split
    head_classes: Tuple[int, ...] = (0, 1, 2, 3, 4)
    tail_classes: Tuple[int, ...] = (5, 6, 7, 8, 9)

    # Public leak (for GAMING_HEAD_ONLY_LEAK)
    leak_fraction: float = 1.0

    # Strategy mix
    benign_frac: float = 0.10
    gaming_frac_main: float = 0.30  # E2's "gaming" condition
    update_scale_factor: float = 1.75  # for update-scaling gaming

    # DP-like noise injection (privacy knob)
    # noise_multiplier = sigma, noise std = sigma * clip_C
    clip_C: float = 1.0
    noise_multipliers: Tuple[float, ...] = (0.00, 0.05, 0.10)  # <-- requested

    # Evaluation summary
    tail_k: int = 7

    seed: int = 42
    root: str = "./data"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG = Config()


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)


# ----------------------------
# 1) Dataset preparation
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = datasets.FashionMNIST(root=CFG.root, train=True, download=True, transform=transform)

# We'll use:
# - 50k for client-local data
# - 10k as public validation candidates (from train)
train_local_size = 50_000
idx_all = np.arange(len(full_train))
idx_local = idx_all[:train_local_size]
idx_public_candidates = idx_all[train_local_size:]

train_local_set = Subset(full_train, idx_local)
targets_all = np.array(full_train.targets)

# Public validation set: head-only from last 10k
head_mask = np.isin(targets_all, list(CFG.head_classes))
public_head_indices = [i for i in idx_public_candidates if head_mask[i]]
public_head_indices = np.array(public_head_indices)

rng = np.random.default_rng(CFG.seed)
rng.shuffle(public_head_indices)

leak_size = int(CFG.leak_fraction * len(public_head_indices))
leak_indices = public_head_indices[:leak_size]

public_val_set = Subset(full_train, public_head_indices)
public_val_loader = DataLoader(public_val_set, batch_size=CFG.batch_size, shuffle=False)

# Leak subset that gaming clients can append
leak_subset = Subset(full_train, leak_indices)


# ----------------------------
# 2) Utilities: partition + filtering
# ----------------------------
def make_dirichlet_partitions(labels: np.ndarray, n_clients: int, alpha: float, rng: np.random.Generator):
    """Returns list of index arrays per client (indices w.r.t. train_local_set)."""
    n_classes = int(labels.max()) + 1
    client_indices: List[List[int]] = [[] for _ in range(n_clients)]

    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue
        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shards = np.split(idx_k, splits[:-1])
        for cid, shard in enumerate(shards):
            client_indices[cid].extend(shard.tolist())

    for cid in range(n_clients):
        rng.shuffle(client_indices[cid])

    return client_indices


def filter_subset_by_labels(base_subset: Subset, allowed_labels: Tuple[int, ...]) -> Subset:
    """base_subset is a Subset of train_local_set; keep only items with labels in allowed_labels."""
    allowed = set(allowed_labels)
    kept = []
    for idx_in_local in base_subset.indices:
        global_idx = train_local_set.indices[idx_in_local]  # index into full_train
        y = int(full_train.targets[global_idx])
        if y in allowed:
            kept.append(idx_in_local)
    return Subset(train_local_set, kept)


def split_subset_train_eval(base_subset: Subset, eval_ratio: float = 0.2) -> Tuple[Subset, Subset]:
    idxs = list(base_subset.indices)
    rng_local = np.random.default_rng(CFG.seed + 123)
    rng_local.shuffle(idxs)
    n_eval = int(round(eval_ratio * len(idxs)))
    eval_idxs = idxs[:n_eval]
    train_idxs = idxs[n_eval:]
    return Subset(train_local_set, train_idxs), Subset(train_local_set, eval_idxs)


def tail_only_subset(base_subset: Subset) -> Subset:
    return filter_subset_by_labels(base_subset, CFG.tail_classes)


def oversample_tail(train_subset: Subset, factor: int = 2) -> ConcatDataset:
    """Benign cooperation: upweight tail samples in local training."""
    tail_sub = tail_only_subset(train_subset)
    if len(tail_sub) == 0:
        return ConcatDataset([train_subset])
    reps = [tail_sub for _ in range(factor)]
    return ConcatDataset([train_subset] + reps)


# Build base client datasets (Dirichlet)
train_targets_local = np.array(full_train.targets)[idx_local]
client_indices = make_dirichlet_partitions(
    labels=train_targets_local,
    n_clients=CFG.n_clients,
    alpha=CFG.dirichlet_alpha,
    rng=np.random.default_rng(CFG.seed)
)

client_base_subsets = [Subset(train_local_set, idxs) for idxs in client_indices]

client_train_subsets: List[Subset] = []
client_eval_subsets: List[Subset] = []
client_tail_eval_subsets: List[Subset] = []
client_tail_eval_sizes: List[int] = []

for cid in range(CFG.n_clients):
    tr, ev = split_subset_train_eval(client_base_subsets[cid], eval_ratio=0.2)
    client_train_subsets.append(tr)
    client_eval_subsets.append(ev)
    tail_ev = tail_only_subset(ev)
    client_tail_eval_subsets.append(tail_ev)
    client_tail_eval_sizes.append(len(tail_ev))


# ----------------------------
# 3) Model
# ----------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def get_model() -> nn.Module:
    return SimpleCNN().to(CFG.device)


# ----------------------------
# 4) Train/Eval helpers
# ----------------------------
@torch.no_grad()
def evaluate_acc(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(CFG.device), y.to(CFG.device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


def local_train(model: nn.Module, dataset, epochs: int) -> nn.Module:
    loader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=True)
    opt = optim.SGD(model.parameters(), lr=CFG.lr, momentum=CFG.momentum)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(CFG.device), y.to(CFG.device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
    return model


def fedavg_from_states(states: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    avg = {}
    for k in states[0].keys():
        stacked = torch.stack([sd[k] for sd in states], dim=0)
        avg[k] = stacked.mean(dim=0)
    return avg


# ----------------------------
# 5) Strategy definitions
# ----------------------------
STR_HONEST = "HONEST"
STR_BENIGN = "BENIGN_TAIL_UPWEIGHT"
STR_GAME_HEAD = "GAMING_HEAD_ONLY_LEAK"
STR_GAME_SCALE = "GAMING_UPDATE_SCALING"


def assign_strategies(n_clients: int,
                      gaming_frac: float,
                      benign_frac: float,
                      rng: np.random.Generator) -> Dict[int, str]:
    """
    Assign each client a strategy from {HONEST, BENIGN, GAMING_HEAD, GAMING_SCALE}.
      - benign_frac portion get BENIGN
      - gaming_frac portion get a GAMING strategy (half head-only, half update-scaling)
      - rest are HONEST
    """
    all_ids = np.arange(n_clients)
    rng.shuffle(all_ids)

    n_benign = int(round(benign_frac * n_clients))
    n_gaming = int(round(gaming_frac * n_clients))

    benign_ids = set(all_ids[:n_benign])
    gaming_ids = list(all_ids[n_benign:n_benign + n_gaming])
    honest_ids = set(all_ids[n_benign + n_gaming:])

    strat = {}
    for cid in benign_ids:
        strat[cid] = STR_BENIGN
    for cid in honest_ids:
        strat[cid] = STR_HONEST

    half = len(gaming_ids) // 2
    for cid in gaming_ids[:half]:
        strat[cid] = STR_GAME_HEAD
    for cid in gaming_ids[half:]:
        strat[cid] = STR_GAME_SCALE

    return strat


def build_client_train_dataset(cid: int, strategy: str):
    base_train = client_train_subsets[cid]

    if strategy == STR_HONEST:
        return base_train

    if strategy == STR_BENIGN:
        return oversample_tail(base_train, factor=2)

    if strategy == STR_GAME_HEAD:
        head_only = filter_subset_by_labels(base_train, CFG.head_classes)
        return ConcatDataset([head_only, leak_subset])

    if strategy == STR_GAME_SCALE:
        return base_train

    raise ValueError(f"Unknown strategy: {strategy}")


# ----------------------------
# 6) DP-like noise injection on transmitted update
# ----------------------------
def _flatten_delta(delta_sd: Dict[str, torch.Tensor]) -> torch.Tensor:
    vecs = []
    for k in delta_sd.keys():
        vecs.append(delta_sd[k].reshape(-1))
    return torch.cat(vecs, dim=0)


def _unflatten_to_sd(vec: torch.Tensor, template_sd: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    offset = 0
    for k in template_sd.keys():
        numel = template_sd[k].numel()
        out[k] = vec[offset:offset + numel].reshape(template_sd[k].shape)
        offset += numel
    return out


def dp_perturb_delta(delta_sd: Dict[str, torch.Tensor], clip_C: float, noise_mult: float) -> Dict[str, torch.Tensor]:
    """
    DP-SGD-like: clip global L2 norm of the update delta, then add Gaussian noise.
    noise std = noise_mult * clip_C
    """
    delta_vec = _flatten_delta(delta_sd)

    # Clip
    norm = torch.norm(delta_vec, p=2)
    if norm > clip_C:
        delta_vec = delta_vec * (clip_C / (norm + 1e-12))

    # Noise
    if noise_mult > 0:
        std = noise_mult * clip_C
        delta_vec = delta_vec + torch.randn_like(delta_vec) * std

    return _unflatten_to_sd(delta_vec, delta_sd)


def make_transmitted_state(global_sd: Dict[str, torch.Tensor],
                           local_sd: Dict[str, torch.Tensor],
                           scale: float,
                           clip_C: float,
                           noise_mult: float) -> Dict[str, torch.Tensor]:
    """
    transmitted = global + DP( scale * (local - global) )
    """
    delta = {}
    for k in global_sd.keys():
        delta[k] = scale * (local_sd[k] - global_sd[k])

    delta = dp_perturb_delta(delta, clip_C=clip_C, noise_mult=noise_mult)

    tx = {}
    for k in global_sd.keys():
        tx[k] = global_sd[k] + delta[k]
    return tx


# ----------------------------
# 7) Run FL once (given gaming_frac and noise_mult)
#    Per-round:
#    - M_t: public head accuracy (metric)
#    - W_t: mean tail accuracy across clients (welfare)
#    - W_std_t: std tail acc across clients
#    - W_min_t, W_max_t: min/max tail acc across clients
#    - tail_nonempty_count_t: number of clients with tail eval > 0
# ----------------------------
def run_once(gaming_frac: float, benign_frac: float, noise_mult: float, seed_offset: int = 0):
    rng = np.random.default_rng(CFG.seed + seed_offset)
    strat_map = assign_strategies(CFG.n_clients, gaming_frac, benign_frac, rng)

    client_train_ds = [build_client_train_dataset(cid, strat_map[cid]) for cid in range(CFG.n_clients)]

    global_model = get_model()
    global_sd = {k: v.detach().clone() for k, v in global_model.state_dict().items()}

    M_hist: List[float] = []
    W_hist: List[float] = []
    W_std_hist: List[float] = []
    W_min_hist: List[float] = []
    W_max_hist: List[float] = []
    tail_nonempty_hist: List[int] = []

    # For reporting how many are gaming/benign/honest (fixed for the run)
    n_honest = sum(1 for s in strat_map.values() if s == STR_HONEST)
    n_benign = sum(1 for s in strat_map.values() if s == STR_BENIGN)
    n_ghead = sum(1 for s in strat_map.values() if s == STR_GAME_HEAD)
    n_gscale = sum(1 for s in strat_map.values() if s == STR_GAME_SCALE)

    for rnd in range(CFG.n_rounds):
        transmitted_states: List[Dict[str, torch.Tensor]] = []

        for cid in range(CFG.n_clients):
            local_model = get_model()
            local_model.load_state_dict(global_sd)

            local_model = local_train(local_model, client_train_ds[cid], epochs=CFG.local_epochs)
            local_sd = local_model.state_dict()

            # scale for UPDATE_SCALING gaming; others scale=1
            if strat_map[cid] == STR_GAME_SCALE and gaming_frac > 0:
                scale = CFG.update_scale_factor
            else:
                scale = 1.0

            # DP-like perturbation on transmitted update (applies to everyone)
            tx_sd = make_transmitted_state(
                global_sd=global_sd,
                local_sd=local_sd,
                scale=scale,
                clip_C=CFG.clip_C,
                noise_mult=noise_mult
            )
            transmitted_states.append(tx_sd)

        # FedAvg update
        global_sd = fedavg_from_states(transmitted_states)
        global_model.load_state_dict(global_sd)

        # Metric: server-observable public head validation accuracy
        M_t = evaluate_acc(global_model, public_val_loader)
        M_hist.append(M_t)

        # Welfare: tail accuracy distribution across clients (experimenter-side evaluation)
        tail_accs = []
        nonempty = 0
        for cid in range(CFG.n_clients):
            tail_ev = client_tail_eval_subsets[cid]
            if len(tail_ev) == 0:
                continue
            nonempty += 1
            loader = DataLoader(tail_ev, batch_size=CFG.batch_size, shuffle=False)
            tail_accs.append(evaluate_acc(global_model, loader))

        tail_nonempty_hist.append(nonempty)

        if len(tail_accs) > 0:
            W_t = float(np.mean(tail_accs))
            W_std_t = float(np.std(tail_accs))
            W_min_t = float(np.min(tail_accs))
            W_max_t = float(np.max(tail_accs))
        else:
            W_t, W_std_t, W_min_t, W_max_t = 0.0, float("nan"), float("nan"), float("nan")

        W_hist.append(W_t)
        W_std_hist.append(W_std_t)
        W_min_hist.append(W_min_t)
        W_max_hist.append(W_max_t)

        if (rnd + 1) % 10 == 0 or rnd == 0:
            print(f"[gf={gaming_frac:.2f} | noise={noise_mult:.2f}] "
                  f"Round {rnd+1:>2}/{CFG.n_rounds} | M(head)={M_t:.4f} | "
                  f"W_tail(mean)={W_t:.4f} | W_std={W_std_t:.4f} | n_tail_clients={nonempty}")

    return {
        "gaming_frac": gaming_frac,
        "benign_frac": benign_frac,
        "noise_mult": noise_mult,
        "strategy_map": strat_map,
        "strategy_counts": {
            "honest": n_honest,
            "benign": n_benign,
            "gaming_head": n_ghead,
            "gaming_scale": n_gscale,
        },
        "M_hist": np.array(M_hist, dtype=np.float32),
        "W_hist": np.array(W_hist, dtype=np.float32),
        "W_std_hist": np.array(W_std_hist, dtype=np.float32),
        "W_min_hist": np.array(W_min_hist, dtype=np.float32),
        "W_max_hist": np.array(W_max_hist, dtype=np.float32),
        "tail_nonempty_hist": np.array(tail_nonempty_hist, dtype=np.int32),
    }


# ----------------------------
# 8) Summary metrics
# ----------------------------
def tail_slice(x: np.ndarray, k: int) -> np.ndarray:
    if len(x) == 0:
        return x
    if len(x) <= k:
        return x
    return x[-k:]


def mean_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.mean(xs)) if len(xs) > 0 else float("nan")


def std_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.std(xs)) if len(xs) > 0 else float("nan")


def min_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.min(xs)) if len(xs) > 0 else float("nan")


def max_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.max(xs)) if len(xs) > 0 else float("nan")


def compute_pog(W_ref: float, W_final: float) -> float:
    if not np.isfinite(W_ref) or W_ref <= 1e-8:
        return float("nan")
    return float((W_ref - W_final) / W_ref)


def safe_div(a: float, b: float, eps: float = 1e-12) -> float:
    return float(a / (b + eps))


# ----------------------------
# 9) Main: Sweep noise levels, compare aligned vs gaming
# ----------------------------
def main():
    print("\n=== E2: Noise / Privacy Trade-off (DP-like update noise) ===")
    print(f"Device: {CFG.device}")
    print(f"Clients={CFG.n_clients}, Rounds={CFG.n_rounds}, LocalEpochs={CFG.local_epochs}")
    print(f"clip_C={CFG.clip_C}, noise_multipliers={CFG.noise_multipliers}")
    print(f"benign_frac={CFG.benign_frac}, gaming_frac_main={CFG.gaming_frac_main}\n")

    # 9.1 Reference baseline (aligned, noise=0)
    print(">> Running reference baseline: gf=0.0, noise=0.0")
    ref = run_once(gaming_frac=0.0, benign_frac=CFG.benign_frac, noise_mult=0.0, seed_offset=0)
    M_ref = mean_last_k(ref["M_hist"], CFG.tail_k)
    W_ref = mean_last_k(ref["W_hist"], CFG.tail_k)

    print("\n[Reference] tail-mean over last K rounds")
    print(f"  M_ref (head metric) = {M_ref:.4f}")
    print(f"  W_ref (tail welfare)= {W_ref:.4f}\n")

    # 9.2 Sweep noise for two conditions: aligned (gf=0) vs gaming (gf=gaming_frac_main)
    results = []
    seed_offset = 10
    for nm in CFG.noise_multipliers:
        # aligned under noise
        print(f"\n>> Running ALIGNED under noise={nm:.2f} (gf=0.0)")
        aligned = run_once(gaming_frac=0.0, benign_frac=CFG.benign_frac, noise_mult=nm, seed_offset=seed_offset)
        seed_offset += 1

        # gaming under noise
        print(f"\n>> Running GAMING under noise={nm:.2f} (gf={CFG.gaming_frac_main:.2f})")
        gaming = run_once(gaming_frac=CFG.gaming_frac_main, benign_frac=CFG.benign_frac, noise_mult=nm, seed_offset=seed_offset)
        seed_offset += 1

        # Compute summary metrics for each condition
        def summarize(tag: str, out: Dict):
            M_final = mean_last_k(out["M_hist"], CFG.tail_k)
            W_final = mean_last_k(out["W_hist"], CFG.tail_k)
            gap = M_final - W_final

            # "Classic" PoG vs reference W_ref (kept as-is; does NOT change your paper's definition)
            pog = compute_pog(W_ref, W_final)

            # Additional metrics (more stable / more informative under noise)
            M_std = std_last_k(out["M_hist"], CFG.tail_k)
            W_std = std_last_k(out["W_hist"], CFG.tail_k)

            # Tail distribution stats across clients (per-round std/min/max, summarized over last K)
            W_std_across_clients_mean = mean_last_k(out["W_std_hist"], CFG.tail_k)
            W_min = mean_last_k(out["W_min_hist"], CFG.tail_k)
            W_max = mean_last_k(out["W_max_hist"], CFG.tail_k)

            # Stability / spread
            range_M = max_last_k(out["M_hist"], CFG.tail_k) - min_last_k(out["M_hist"], CFG.tail_k)
            range_W = max_last_k(out["W_hist"], CFG.tail_k) - min_last_k(out["W_hist"], CFG.tail_k)

            # Deltas vs reference
            dM_vs_ref = M_final - M_ref
            dW_vs_ref = W_final - W_ref

            # Normalized gap ratio (how large is gap relative to welfare)
            gap_ratio = safe_div(gap, max(W_final, 0.0))

            # Average number of clients with non-empty tail eval (should be constant, but log anyway)
            tail_nonempty = int(np.round(mean_last_k(out["tail_nonempty_hist"].astype(np.float32), CFG.tail_k)))

            sc = out["strategy_counts"]

            return {
                "condition": tag,
                "noise_mult": float(out["noise_mult"]),
                "gaming_frac": float(out["gaming_frac"]),
                "benign_frac": float(out["benign_frac"]),
                "n_clients": int(CFG.n_clients),
                "n_honest": int(sc["honest"]),
                "n_benign": int(sc["benign"]),
                "n_gaming_head": int(sc["gaming_head"]),
                "n_gaming_scale": int(sc["gaming_scale"]),

                # Primary end metrics
                "M_final": float(M_final),
                "W_tail_final": float(W_final),
                "gap_M_minus_W": float(gap),
                "gap_ratio_gap_over_W": float(gap_ratio),
                "PoG_vs_ref": float(pog),

                # Deltas vs reference
                "M_delta_vs_ref": float(dM_vs_ref),
                "W_delta_vs_ref": float(dW_vs_ref),

                # Within-run temporal stability (last K rounds)
                "M_std_lastK": float(M_std),
                "W_std_lastK": float(W_std),
                "M_range_lastK": float(range_M),
                "W_range_lastK": float(range_W),

                # Across-client tail distribution stats (last K rounds averaged)
                "W_std_across_clients_lastK": float(W_std_across_clients_mean),
                "W_min_across_clients_lastK": float(W_min),
                "W_max_across_clients_lastK": float(W_max),

                # Tail eval coverage
                "tail_nonempty_clients_lastK": int(tail_nonempty),
            }

        row_aligned = summarize("ALIGNED", aligned)
        row_gaming = summarize("GAMING", gaming)

        # Paired "extra effect" metrics at the same noise (does NOT change PoG definition)
        # These help you talk about "additional harm under gaming" without redefining PoG.
        deltaW_same_noise = row_aligned["W_tail_final"] - row_gaming["W_tail_final"]
        deltaGap_same_noise = row_gaming["gap_M_minus_W"] - row_aligned["gap_M_minus_W"]

        # Relative additional harm ratio (if aligned welfare is too small, it will blow up; keep it but be cautious)
        rel_deltaW = safe_div(deltaW_same_noise, max(row_aligned["W_tail_final"], 0.0))

        # Store rows with paired deltas
        row_aligned["paired_deltaW_aligned_minus_gaming"] = float(deltaW_same_noise)
        row_aligned["paired_deltaGap_gaming_minus_aligned"] = float(deltaGap_same_noise)
        row_aligned["paired_rel_deltaW_over_Waligned"] = float(rel_deltaW)

        row_gaming["paired_deltaW_aligned_minus_gaming"] = float(deltaW_same_noise)
        row_gaming["paired_deltaGap_gaming_minus_aligned"] = float(deltaGap_same_noise)
        row_gaming["paired_rel_deltaW_over_Waligned"] = float(rel_deltaW)

        results.append(row_aligned)
        results.append(row_gaming)

        # quick print
        print(f"\n[Summary @ noise={nm:.2f}]")
        print(f"  ALIGNED: M={row_aligned['M_final']:.4f} | W={row_aligned['W_tail_final']:.4f} | "
              f"gap={row_aligned['gap_M_minus_W']:.4f} | PoG(ref)={row_aligned['PoG_vs_ref']:.4f}")
        print(f"  GAMING : M={row_gaming['M_final']:.4f} | W={row_gaming['W_tail_final']:.4f} | "
              f"gap={row_gaming['gap_M_minus_W']:.4f} | PoG(ref)={row_gaming['PoG_vs_ref']:.4f}")
        print(f"  Paired: ΔW(aligned-gaming)={deltaW_same_noise:+.4f} | "
              f"Δgap(gaming-aligned)={deltaGap_same_noise:+.4f} | relΔW={rel_deltaW:+.4f}")

    # 9.3 Save CSV
    out_path = "E2_noise_privacy_tradeoff_results.csv"
    import csv
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        fieldnames = list(results[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    print(f"\nSaved: {out_path}")
    print("Done.")


if __name__ == "__main__":
    main()


=== E2: Noise / Privacy Trade-off (DP-like update noise) ===
Device: cuda
Clients=12, Rounds=25, LocalEpochs=1
clip_C=1.0, noise_multipliers=(0.0, 0.05, 0.1)
benign_frac=0.1, gaming_frac_main=0.3

>> Running reference baseline: gf=0.0, noise=0.0
[gf=0.00 | noise=0.00] Round  1/25 | M(head)=0.4477 | W_tail(mean)=0.4587 | W_std=0.1238 | n_tail_clients=12
[gf=0.00 | noise=0.00] Round 10/25 | M(head)=0.8371 | W_tail(mean)=0.8291 | W_std=0.0725 | n_tail_clients=12
[gf=0.00 | noise=0.00] Round 20/25 | M(head)=0.8690 | W_tail(mean)=0.8483 | W_std=0.0610 | n_tail_clients=12

[Reference] tail-mean over last K rounds
  M_ref (head metric) = 0.8605
  W_ref (tail welfare)= 0.8767


>> Running ALIGNED under noise=0.00 (gf=0.0)
[gf=0.00 | noise=0.00] Round  1/25 | M(head)=0.5039 | W_tail(mean)=0.5282 | W_std=0.1274 | n_tail_clients=12
[gf=0.00 | noise=0.00] Round 10/25 | M(head)=0.8389 | W_tail(mean)=0.8468 | W_std=0.0602 | n_tail_clients=12
[gf=0.00 | noise=0.00] Round 20/25 | M(head)=0.8613 | W_t

# High-alignment metrics

In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms


# ============================================================
# E3: High-alignment regime (metric closer to welfare)
# Fashion-MNIST, same skeleton as your E2 script
#
# Key change vs E2:
# - Remove DP noise sweep
# - Sweep lambda for mixed public metric:
#     M_public(lambda) = (1-lambda)*M_head + lambda*M_tail
# - Keep strategy mix / partitions / model / rounds identical
# - Save CSV with richer metrics (similar to E2)
# ============================================================

# ----------------------------
# 0) Config / Seed
# ----------------------------
@dataclass
class Config:
    # Keep small for Colab
    n_clients: int = 12
    n_rounds: int = 25
    local_epochs: int = 1
    batch_size: int = 64
    lr: float = 0.01
    momentum: float = 0.9

    # Non-IID partition
    dirichlet_alpha: float = 0.5

    # Head/Tail split
    head_classes: Tuple[int, ...] = (0, 1, 2, 3, 4)
    tail_classes: Tuple[int, ...] = (5, 6, 7, 8, 9)

    # Public leak (for GAMING_HEAD_ONLY_LEAK)
    leak_fraction: float = 1.0

    # Strategy mix (same as your E2)
    benign_frac: float = 0.10
    gaming_frac_main: float = 0.30  # E3's "gaming" condition
    update_scale_factor: float = 1.75  # for update-scaling gaming

    # E3: alignment knob(s)
    # lambda=0 -> metric=head only, lambda=1 -> metric=tail only
    metric_lambdas: Tuple[float, ...] = (0.00, 0.30, 0.60)

    # Evaluation summary
    tail_k: int = 7  # last K rounds to average

    seed: int = 42
    root: str = "./data"
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG = Config()


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)


# ----------------------------
# 1) Dataset preparation
# ----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = datasets.FashionMNIST(root=CFG.root, train=True, download=True, transform=transform)

# We'll use:
# - 50k for client-local data
# - 10k as public validation candidates (from train)
train_local_size = 50_000
idx_all = np.arange(len(full_train))
idx_local = idx_all[:train_local_size]
idx_public_candidates = idx_all[train_local_size:]

train_local_set = Subset(full_train, idx_local)
targets_all = np.array(full_train.targets)

# Build public validation sets (head + tail) from last 10k
head_mask = np.isin(targets_all, list(CFG.head_classes))
tail_mask = np.isin(targets_all, list(CFG.tail_classes))

public_head_indices = np.array([i for i in idx_public_candidates if head_mask[i]])
public_tail_indices = np.array([i for i in idx_public_candidates if tail_mask[i]])

rng = np.random.default_rng(CFG.seed)
rng.shuffle(public_head_indices)
rng.shuffle(public_tail_indices)

public_head_val_set = Subset(full_train, public_head_indices)
public_tail_val_set = Subset(full_train, public_tail_indices)

public_head_val_loader = DataLoader(public_head_val_set, batch_size=CFG.batch_size, shuffle=False)
public_tail_val_loader = DataLoader(public_tail_val_set, batch_size=CFG.batch_size, shuffle=False)

# Leak subset that gaming clients can append (head-only, as in your E2)
leak_size = int(CFG.leak_fraction * len(public_head_indices))
leak_indices = public_head_indices[:leak_size]
leak_subset = Subset(full_train, leak_indices)


# ----------------------------
# 2) Utilities: partition + filtering
# ----------------------------
def make_dirichlet_partitions(labels: np.ndarray, n_clients: int, alpha: float, rng: np.random.Generator):
    """Returns list of index arrays per client (indices w.r.t. train_local_set)."""
    n_classes = int(labels.max()) + 1
    client_indices: List[List[int]] = [[] for _ in range(n_clients)]

    class_indices = []
    for k in range(n_classes):
        idx_k = np.where(labels == k)[0]
        rng.shuffle(idx_k)
        class_indices.append(idx_k)

    for k in range(n_classes):
        idx_k = class_indices[k]
        n_k = len(idx_k)
        if n_k == 0:
            continue
        proportions = rng.dirichlet(alpha * np.ones(n_clients))
        proportions = proportions / proportions.sum()
        splits = (np.cumsum(proportions) * n_k).astype(int)
        shards = np.split(idx_k, splits[:-1])
        for cid, shard in enumerate(shards):
            client_indices[cid].extend(shard.tolist())

    for cid in range(n_clients):
        rng.shuffle(client_indices[cid])

    return client_indices


def filter_subset_by_labels(base_subset: Subset, allowed_labels: Tuple[int, ...]) -> Subset:
    """base_subset is a Subset of train_local_set; keep only items with labels in allowed_labels."""
    allowed = set(allowed_labels)
    kept = []
    for idx_in_local in base_subset.indices:
        global_idx = train_local_set.indices[idx_in_local]  # index into full_train
        y = int(full_train.targets[global_idx])
        if y in allowed:
            kept.append(idx_in_local)
    return Subset(train_local_set, kept)


def split_subset_train_eval(base_subset: Subset, eval_ratio: float = 0.2) -> Tuple[Subset, Subset]:
    idxs = list(base_subset.indices)
    rng_local = np.random.default_rng(CFG.seed + 123)
    rng_local.shuffle(idxs)
    n_eval = int(round(eval_ratio * len(idxs)))
    eval_idxs = idxs[:n_eval]
    train_idxs = idxs[n_eval:]
    return Subset(train_local_set, train_idxs), Subset(train_local_set, eval_idxs)


def tail_only_subset(base_subset: Subset) -> Subset:
    return filter_subset_by_labels(base_subset, CFG.tail_classes)


def oversample_tail(train_subset: Subset, factor: int = 2) -> ConcatDataset:
    """Benign cooperation: upweight tail samples in local training."""
    tail_sub = tail_only_subset(train_subset)
    if len(tail_sub) == 0:
        return ConcatDataset([train_subset])
    reps = [tail_sub for _ in range(factor)]
    return ConcatDataset([train_subset] + reps)


# Build base client datasets (Dirichlet)
train_targets_local = np.array(full_train.targets)[idx_local]
client_indices = make_dirichlet_partitions(
    labels=train_targets_local,
    n_clients=CFG.n_clients,
    alpha=CFG.dirichlet_alpha,
    rng=np.random.default_rng(CFG.seed)
)

client_base_subsets = [Subset(train_local_set, idxs) for idxs in client_indices]

client_train_subsets: List[Subset] = []
client_eval_subsets: List[Subset] = []
client_tail_eval_subsets: List[Subset] = []
client_tail_eval_sizes: List[int] = []

for cid in range(CFG.n_clients):
    tr, ev = split_subset_train_eval(client_base_subsets[cid], eval_ratio=0.2)
    client_train_subsets.append(tr)
    client_eval_subsets.append(ev)
    tail_ev = tail_only_subset(ev)
    client_tail_eval_subsets.append(tail_ev)
    client_tail_eval_sizes.append(len(tail_ev))


# ----------------------------
# 3) Model
# ----------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


def get_model() -> nn.Module:
    return SimpleCNN().to(CFG.device)


# ----------------------------
# 4) Train/Eval helpers
# ----------------------------
@torch.no_grad()
def evaluate_acc(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(CFG.device), y.to(CFG.device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


def local_train(model: nn.Module, dataset, epochs: int) -> nn.Module:
    loader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=True)
    opt = optim.SGD(model.parameters(), lr=CFG.lr, momentum=CFG.momentum)
    crit = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(CFG.device), y.to(CFG.device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
    return model


def fedavg_from_states(states: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    avg = {}
    for k in states[0].keys():
        stacked = torch.stack([sd[k] for sd in states], dim=0)
        avg[k] = stacked.mean(dim=0)
    return avg


# ----------------------------
# 5) Strategy definitions (same as E2)
# ----------------------------
STR_HONEST = "HONEST"
STR_BENIGN = "BENIGN_TAIL_UPWEIGHT"
STR_GAME_HEAD = "GAMING_HEAD_ONLY_LEAK"
STR_GAME_SCALE = "GAMING_UPDATE_SCALING"


def assign_strategies(n_clients: int,
                      gaming_frac: float,
                      benign_frac: float,
                      rng: np.random.Generator) -> Dict[int, str]:
    """
    Assign each client a strategy from {HONEST, BENIGN, GAMING_HEAD, GAMING_SCALE}.
      - benign_frac portion get BENIGN
      - gaming_frac portion get a GAMING strategy (half head-only, half update-scaling)
      - rest are HONEST
    """
    all_ids = np.arange(n_clients)
    rng.shuffle(all_ids)

    n_benign = int(round(benign_frac * n_clients))
    n_gaming = int(round(gaming_frac * n_clients))

    benign_ids = set(all_ids[:n_benign])
    gaming_ids = list(all_ids[n_benign:n_benign + n_gaming])
    honest_ids = set(all_ids[n_benign + n_gaming:])

    strat = {}
    for cid in benign_ids:
        strat[cid] = STR_BENIGN
    for cid in honest_ids:
        strat[cid] = STR_HONEST

    half = len(gaming_ids) // 2
    for cid in gaming_ids[:half]:
        strat[cid] = STR_GAME_HEAD
    for cid in gaming_ids[half:]:
        strat[cid] = STR_GAME_SCALE

    return strat


def build_client_train_dataset(cid: int, strategy: str):
    base_train = client_train_subsets[cid]

    if strategy == STR_HONEST:
        return base_train

    if strategy == STR_BENIGN:
        return oversample_tail(base_train, factor=2)

    if strategy == STR_GAME_HEAD:
        head_only = filter_subset_by_labels(base_train, CFG.head_classes)
        return ConcatDataset([head_only, leak_subset])

    if strategy == STR_GAME_SCALE:
        return base_train

    raise ValueError(f"Unknown strategy: {strategy}")


# ----------------------------
# 6) Transmitted state (no DP in E3; keep update scaling for gaming)
# ----------------------------
def make_transmitted_state_no_dp(global_sd: Dict[str, torch.Tensor],
                                 local_sd: Dict[str, torch.Tensor],
                                 scale: float) -> Dict[str, torch.Tensor]:
    """
    transmitted = global + scale * (local - global)
    """
    tx = {}
    for k in global_sd.keys():
        tx[k] = global_sd[k] + scale * (local_sd[k] - global_sd[k])
    return tx


# ----------------------------
# 7) Run FL once (given gaming_frac and lambda)
#    Per-round:
#    - M_head_t: public head accuracy
#    - M_tail_t: public tail accuracy
#    - M_pub_t : mixed metric (lambda)
#    - W_t     : mean tail accuracy across clients (welfare)
#    plus distribution stats across clients on tail eval
# ----------------------------
def run_once(gaming_frac: float, benign_frac: float, metric_lambda: float, seed_offset: int = 0):
    rng = np.random.default_rng(CFG.seed + seed_offset)
    strat_map = assign_strategies(CFG.n_clients, gaming_frac, benign_frac, rng)

    client_train_ds = [build_client_train_dataset(cid, strat_map[cid]) for cid in range(CFG.n_clients)]

    global_model = get_model()
    global_sd = {k: v.detach().clone() for k, v in global_model.state_dict().items()}

    M_head_hist: List[float] = []
    M_tail_hist: List[float] = []
    M_pub_hist: List[float] = []

    W_hist: List[float] = []
    W_std_hist: List[float] = []
    W_min_hist: List[float] = []
    W_max_hist: List[float] = []
    tail_nonempty_hist: List[int] = []

    n_honest = sum(1 for s in strat_map.values() if s == STR_HONEST)
    n_benign = sum(1 for s in strat_map.values() if s == STR_BENIGN)
    n_ghead = sum(1 for s in strat_map.values() if s == STR_GAME_HEAD)
    n_gscale = sum(1 for s in strat_map.values() if s == STR_GAME_SCALE)

    for rnd in range(CFG.n_rounds):
        transmitted_states: List[Dict[str, torch.Tensor]] = []

        for cid in range(CFG.n_clients):
            local_model = get_model()
            local_model.load_state_dict(global_sd)

            local_model = local_train(local_model, client_train_ds[cid], epochs=CFG.local_epochs)
            local_sd = local_model.state_dict()

            # scale for UPDATE_SCALING gaming; others scale=1
            if strat_map[cid] == STR_GAME_SCALE and gaming_frac > 0:
                scale = CFG.update_scale_factor
            else:
                scale = 1.0

            tx_sd = make_transmitted_state_no_dp(global_sd=global_sd, local_sd=local_sd, scale=scale)
            transmitted_states.append(tx_sd)

        # FedAvg update
        global_sd = fedavg_from_states(transmitted_states)
        global_model.load_state_dict(global_sd)

        # Public accuracies
        M_head = evaluate_acc(global_model, public_head_val_loader)
        M_tail = evaluate_acc(global_model, public_tail_val_loader)
        M_pub = (1.0 - metric_lambda) * M_head + metric_lambda * M_tail

        M_head_hist.append(M_head)
        M_tail_hist.append(M_tail)
        M_pub_hist.append(M_pub)

        # Welfare: tail accuracy distribution across clients (experimenter-side evaluation)
        tail_accs = []
        nonempty = 0
        for cid in range(CFG.n_clients):
            tail_ev = client_tail_eval_subsets[cid]
            if len(tail_ev) == 0:
                continue
            nonempty += 1
            loader = DataLoader(tail_ev, batch_size=CFG.batch_size, shuffle=False)
            tail_accs.append(evaluate_acc(global_model, loader))

        tail_nonempty_hist.append(nonempty)

        if len(tail_accs) > 0:
            W_t = float(np.mean(tail_accs))
            W_std_t = float(np.std(tail_accs))
            W_min_t = float(np.min(tail_accs))
            W_max_t = float(np.max(tail_accs))
        else:
            W_t, W_std_t, W_min_t, W_max_t = 0.0, float("nan"), float("nan"), float("nan")

        W_hist.append(W_t)
        W_std_hist.append(W_std_t)
        W_min_hist.append(W_min_t)
        W_max_hist.append(W_max_t)

        if (rnd + 1) % 10 == 0 or rnd == 0:
            print(f"[gf={gaming_frac:.2f} | lam={metric_lambda:.2f}] "
                  f"Round {rnd+1:>2}/{CFG.n_rounds} | "
                  f"M_head={M_head:.4f} | M_tail={M_tail:.4f} | M_pub={M_pub:.4f} | "
                  f"W_tail(mean)={W_t:.4f} | W_std={W_std_t:.4f} | n_tail_clients={nonempty}")

    return {
        "gaming_frac": gaming_frac,
        "benign_frac": benign_frac,
        "metric_lambda": metric_lambda,
        "strategy_map": strat_map,
        "strategy_counts": {
            "honest": n_honest,
            "benign": n_benign,
            "gaming_head": n_ghead,
            "gaming_scale": n_gscale,
        },
        "M_head_hist": np.array(M_head_hist, dtype=np.float32),
        "M_tail_hist": np.array(M_tail_hist, dtype=np.float32),
        "M_pub_hist": np.array(M_pub_hist, dtype=np.float32),
        "W_hist": np.array(W_hist, dtype=np.float32),
        "W_std_hist": np.array(W_std_hist, dtype=np.float32),
        "W_min_hist": np.array(W_min_hist, dtype=np.float32),
        "W_max_hist": np.array(W_max_hist, dtype=np.float32),
        "tail_nonempty_hist": np.array(tail_nonempty_hist, dtype=np.int32),
    }


# ----------------------------
# 8) Summary metrics (same style as E2)
# ----------------------------
def tail_slice(x: np.ndarray, k: int) -> np.ndarray:
    if len(x) == 0:
        return x
    if len(x) <= k:
        return x
    return x[-k:]


def mean_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.mean(xs)) if len(xs) > 0 else float("nan")


def std_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.std(xs)) if len(xs) > 0 else float("nan")


def min_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.min(xs)) if len(xs) > 0 else float("nan")


def max_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.max(xs)) if len(xs) > 0 else float("nan")


def safe_div(a: float, b: float, eps: float = 1e-12) -> float:
    return float(a / (b + eps))


# "PoG-style" summary helpers.
# Since your paper's exact PoG definition may differ, we compute two common views:
# (1) Global reference vs aligned @ lambda=0 (kept constant across lambdas)
# (2) Paired at same lambda: (W_aligned - W_gaming) / W_aligned  (good for E3 story)
def pog_vs_ref(W_ref: float, W_final: float) -> float:
    if not np.isfinite(W_ref) or W_ref <= 1e-8:
        return float("nan")
    return float((W_ref - W_final) / W_ref)


def pog_paired_same_lambda(W_aligned: float, W_gaming: float) -> float:
    if not np.isfinite(W_aligned) or W_aligned <= 1e-8:
        return float("nan")
    return float((W_aligned - W_gaming) / W_aligned)


# ----------------------------
# 9) Main: Sweep lambdas, compare aligned vs gaming
# ----------------------------
def main():
    print("\n=== E3: High-alignment regime (Fashion-MNIST) ===")
    print(f"Device: {CFG.device}")
    print(f"Clients={CFG.n_clients}, Rounds={CFG.n_rounds}, LocalEpochs={CFG.local_epochs}")
    print(f"dirichlet_alpha={CFG.dirichlet_alpha}")
    print(f"benign_frac={CFG.benign_frac}, gaming_frac_main={CFG.gaming_frac_main}")
    print(f"metric_lambdas={CFG.metric_lambdas}")
    print(f"Head classes={CFG.head_classes} | Tail classes={CFG.tail_classes}\n")

    # 9.1 Global reference baseline (aligned, lambda=0.0) to keep a fixed anchor if desired
    print(">> Running global reference baseline: ALIGNED, lambda=0.0")
    ref = run_once(gaming_frac=0.0, benign_frac=CFG.benign_frac, metric_lambda=0.0, seed_offset=0)
    Mpub_ref = mean_last_k(ref["M_pub_hist"], CFG.tail_k)
    W_ref = mean_last_k(ref["W_hist"], CFG.tail_k)

    print("\n[Global Reference] mean over last K rounds")
    print(f"  M_pub_ref (lambda=0.0) = {Mpub_ref:.4f}")
    print(f"  W_ref (tail welfare)   = {W_ref:.4f}\n")

    results = []
    seed_offset = 10

    for lam in CFG.metric_lambdas:
        print(f"\n==============================")
        print(f" Lambda = {lam:.2f}")
        print(f"==============================")

        # ALIGNED at this lambda
        print(f">> Running ALIGNED @ lambda={lam:.2f} (gf=0.0)")
        aligned = run_once(gaming_frac=0.0, benign_frac=CFG.benign_frac, metric_lambda=lam, seed_offset=seed_offset)
        seed_offset += 1

        # GAMING at this lambda
        print(f"\n>> Running GAMING  @ lambda={lam:.2f} (gf={CFG.gaming_frac_main:.2f})")
        gaming = run_once(gaming_frac=CFG.gaming_frac_main, benign_frac=CFG.benign_frac, metric_lambda=lam, seed_offset=seed_offset)
        seed_offset += 1

        def summarize(tag: str, out: Dict):
            M_head_final = mean_last_k(out["M_head_hist"], CFG.tail_k)
            M_tail_final = mean_last_k(out["M_tail_hist"], CFG.tail_k)
            M_pub_final = mean_last_k(out["M_pub_hist"], CFG.tail_k)
            W_final = mean_last_k(out["W_hist"], CFG.tail_k)

            # gaps
            gap_pub_minus_W = M_pub_final - W_final
            gap_head_minus_tail = M_head_final - M_tail_final
            align_error_abs = abs(gap_pub_minus_W)

            # global reference PoG-style (anchor at aligned lambda=0.0 reference)
            pog_global = pog_vs_ref(W_ref=W_ref, W_final=W_final)

            # stability
            Mpub_std = std_last_k(out["M_pub_hist"], CFG.tail_k)
            W_std = std_last_k(out["W_hist"], CFG.tail_k)

            Mpub_range = max_last_k(out["M_pub_hist"], CFG.tail_k) - min_last_k(out["M_pub_hist"], CFG.tail_k)
            W_range = max_last_k(out["W_hist"], CFG.tail_k) - min_last_k(out["W_hist"], CFG.tail_k)

            # across-client tail distribution stats (already per-round)
            W_std_across_clients = mean_last_k(out["W_std_hist"], CFG.tail_k)
            W_min = mean_last_k(out["W_min_hist"], CFG.tail_k)
            W_max = mean_last_k(out["W_max_hist"], CFG.tail_k)

            # normalized ratios
            gap_pub_over_W = safe_div(gap_pub_minus_W, max(W_final, 0.0))
            gap_headtail_over_tail = safe_div(gap_head_minus_tail, max(M_tail_final, 0.0))

            # tail coverage
            tail_nonempty = int(np.round(mean_last_k(out["tail_nonempty_hist"].astype(np.float32), CFG.tail_k)))

            sc = out["strategy_counts"]

            return {
                "condition": tag,
                "lambda": float(out["metric_lambda"]),
                "gaming_frac": float(out["gaming_frac"]),
                "benign_frac": float(out["benign_frac"]),
                "n_clients": int(CFG.n_clients),

                "n_honest": int(sc["honest"]),
                "n_benign": int(sc["benign"]),
                "n_gaming_head": int(sc["gaming_head"]),
                "n_gaming_scale": int(sc["gaming_scale"]),

                # Primary: public metrics + welfare
                "M_head_final": float(M_head_final),
                "M_tail_final": float(M_tail_final),
                "M_pub_final": float(M_pub_final),
                "W_tail_final": float(W_final),

                # Alignment / gap diagnostics
                "gap_Mpub_minus_W": float(gap_pub_minus_W),
                "abs_align_error_|Mpub-W|": float(align_error_abs),
                "gap_Mhead_minus_Mtail": float(gap_head_minus_tail),

                "gap_pub_over_W": float(gap_pub_over_W),
                "gap_headtail_over_Mtail": float(gap_headtail_over_tail),

                # PoG-style (global anchor; keep constant anchor)
                "PoG_vs_global_refW": float(pog_global),

                # Temporal stability (last K)
                "Mpub_std_lastK": float(Mpub_std),
                "W_std_lastK": float(W_std),
                "Mpub_range_lastK": float(Mpub_range),
                "W_range_lastK": float(W_range),

                # Across-client tail distribution stats (last K averaged)
                "W_std_across_clients_lastK": float(W_std_across_clients),
                "W_min_across_clients_lastK": float(W_min),
                "W_max_across_clients_lastK": float(W_max),

                # Tail eval coverage
                "tail_nonempty_clients_lastK": int(tail_nonempty),
            }

        row_aligned = summarize("ALIGNED", aligned)
        row_gaming = summarize("GAMING", gaming)

        # Paired, same-lambda effects (this is the clean E3 story)
        paired_deltaW = row_aligned["W_tail_final"] - row_gaming["W_tail_final"]
        paired_deltaMpub = row_gaming["M_pub_final"] - row_aligned["M_pub_final"]
        paired_deltaGap = row_gaming["gap_Mpub_minus_W"] - row_aligned["gap_Mpub_minus_W"]
        paired_pog = pog_paired_same_lambda(row_aligned["W_tail_final"], row_gaming["W_tail_final"])

        # Store paired columns on both rows
        for r in (row_aligned, row_gaming):
            r["paired_deltaW_aligned_minus_gaming"] = float(paired_deltaW)
            r["paired_deltaMpub_gaming_minus_aligned"] = float(paired_deltaMpub)
            r["paired_deltaGap_gaming_minus_aligned"] = float(paired_deltaGap)
            r["paired_PoG_same_lambda"] = float(paired_pog)

        results.append(row_aligned)
        results.append(row_gaming)

        # quick print
        print(f"\n[Summary @ lambda={lam:.2f}] (last K mean)")
        print(f"  ALIGNED: Mpub={row_aligned['M_pub_final']:.4f} | W={row_aligned['W_tail_final']:.4f} | "
              f"gap={row_aligned['gap_Mpub_minus_W']:.4f}")
        print(f"  GAMING : Mpub={row_gaming['M_pub_final']:.4f} | W={row_gaming['W_tail_final']:.4f} | "
              f"gap={row_gaming['gap_Mpub_minus_W']:.4f}")
        print(f"  Paired : ΔW(aligned-gaming)={paired_deltaW:+.4f} | "
              f"Δgap(gaming-aligned)={paired_deltaGap:+.4f} | paired_PoG={paired_pog:+.4f}")

    # 9.2 Save CSV
    out_path = "E3_high_alignment_metric_sweep_results.csv"
    import csv
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        fieldnames = list(results[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

    print(f"\nSaved: {out_path}")
    print("Done.")


if __name__ == "__main__":
    main()


=== E3: High-alignment regime (Fashion-MNIST) ===
Device: cuda
Clients=12, Rounds=25, LocalEpochs=1
dirichlet_alpha=0.5
benign_frac=0.1, gaming_frac_main=0.3
metric_lambdas=(0.0, 0.3, 0.6)
Head classes=(0, 1, 2, 3, 4) | Tail classes=(5, 6, 7, 8, 9)

>> Running global reference baseline: ALIGNED, lambda=0.0
[gf=0.00 | lam=0.00] Round  1/25 | M_head=0.6344 | M_tail=0.5729 | M_pub=0.6344 | W_tail(mean)=0.5971 | W_std=0.1427 | n_tail_clients=12
[gf=0.00 | lam=0.00] Round 10/25 | M_head=0.8485 | M_tail=0.8365 | M_pub=0.8485 | W_tail(mean)=0.8358 | W_std=0.0664 | n_tail_clients=12
[gf=0.00 | lam=0.00] Round 20/25 | M_head=0.8682 | M_tail=0.8676 | M_pub=0.8682 | W_tail(mean)=0.8692 | W_std=0.0552 | n_tail_clients=12

[Global Reference] mean over last K rounds
  M_pub_ref (lambda=0.0) = 0.8648
  W_ref (tail welfare)   = 0.8791


 Lambda = 0.00
>> Running ALIGNED @ lambda=0.00 (gf=0.0)
[gf=0.00 | lam=0.00] Round  1/25 | M_head=0.7277 | M_tail=0.6208 | M_pub=0.7277 | W_tail(mean)=0.6281 | W_std

# Modern attack--defense replication

In [ ]:
!pip -q install flwr-datasets[vision] datasets

In [ ]:
# ============================================================
# E4 FULL (No LEAF): FEMNIST via HuggingFace/Flower Datasets
#   - Attacks: PoisonedFL (multi-round consistency + proximal + dyn magnitude),
#              Backdoor/Model Replacement
#   - Defenses: FedCC (linear CKA filtering), Attack-Adaptive Aggregation
#   - FL: FedAvg skeleton (with defense hooks)
#   - Output: CSV + NPZ histories
# ============================================================

# If running on Colab, install once:
# !pip -q install flwr-datasets[vision] datasets

import os
import csv
import json
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset


# ----------------------------
# 0) Config / Seed
# ----------------------------
from dataclasses import dataclass
from typing import Tuple
import torch
import os

@dataclass
class Config:
    # compute budget (Colab-friendly)
    n_clients: int = 32            # CHANGED (24 -> 32)
    n_rounds: int = 100            # CHANGED (60 -> 100)
    local_epochs: int = 1
    batch_size: int = 64
    lr: float = 0.02
    momentum: float = 0.9
    weight_decay: float = 0.0

    # participation
    clients_per_round: int = 12    # CHANGED (8 -> 12)

    # FEMNIST classes: typically 62 classes (digits + letters)
    # We'll define "head" as digits, "tail" as letters for head/tail gap.
    head_classes: Tuple[int, ...] = tuple(range(0, 10))       # digits
    tail_classes: Tuple[int, ...] = tuple(range(10, 62))      # letters

    # public sets (sampled from client train data)
    public_support_size: int = 128       # (CKA용) 유지
    public_head_eval_size: int = 4096    # CHANGED (2048 -> 4096)
    public_tail_eval_size: int = 4096    # CHANGED (2048 -> 4096)

    # malicious population + attack schedule
    malicious_frac: float = 0.30         # CHANGED (0.20 -> 0.30)
    attack_start_round: int = 5          # CHANGED (8 -> 5)

    # PoisonedFL-ish knobs (more faithful but still light)
    poisonedfl_scale: float = 2.2
    poisonedfl_scale_min: float = 1.0
    poisonedfl_scale_max: float = 4.0
    poisonedfl_dyn_eta: float = 0.8
    poisonedfl_beta_consistency: float = 1e-3
    poisonedfl_mu_global: float = 5e-4
    poisonedfl_tail_label_flip: bool = True

    # Backdoor / Model Replacement knobs
    backdoor_target_label: int = 0
    backdoor_poison_frac: float = 0.30
    model_replacement_gamma: float = 6.0
    trigger_size: int = 3
    trigger_value: float = 1.0   # in normalized space, clamped to [-1, 1]

    # FedCC (CKA) filtering knobs
    fedcc_reject_frac: float = 0.25
    fedcc_min_keep: int = 3

    # Attack-adaptive aggregation knobs
    adaagg_alpha: float = 3.0

    # summary / alarm
    tail_k: int = 10
    warmup_ignore: int = 5
    alarm_threshold_std: float = 2.0     # CHANGED (3.0 -> 2.0)

    # FEMNIST client selection safeguards
    min_train_samples_per_client: int = 40
    min_test_samples_per_client: int = 10
    max_partition_id_try: int = 5000

    # reproducibility
    seed: int = 42
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # outputs
    out_dir: str = "./E4_outputs_noLEAF"
    csv_name: str = "E4_FEMNIST_2x2_results.csv"


CFG = Config()
os.makedirs(CFG.out_dir, exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)


# ----------------------------
# 1) FEMNIST via HF/Flower Datasets
# ----------------------------
def build_femnist_clients_from_hf(
    n_clients: int,
    train_ratio: float,
    seed: int,
    min_train: int,
    min_test: int,
    max_try: int,
) -> Tuple[List[TensorDataset], List[TensorDataset], int]:
    """
    Uses flwr_datasets FederatedDataset with NaturalIdPartitioner(partition_by="writer_id").
    Loads partitions sequentially (partition_id=0,1,2,...) until collecting enough clients.

    Returns:
      client_train_sets, client_test_sets, num_classes (assumed 62 if present; otherwise inferred)
    """
    from flwr_datasets import FederatedDataset
    from flwr_datasets.partitioner import NaturalIdPartitioner

    rng = np.random.default_rng(seed)

    fds = FederatedDataset(
        dataset="flwrlabs/femnist",
        partitioners={"train": NaturalIdPartitioner(partition_by="writer_id")},
    )

    client_train_sets: List[TensorDataset] = []
    client_test_sets: List[TensorDataset] = []
    max_label = -1

    # Helper: PIL->torch normalized [-1,1]
    def to_tensors(part) -> Tuple[torch.Tensor, torch.Tensor]:
        images = part["image"]  # PIL images
        labels = np.array(part["character"], dtype=np.int64)
        # PIL -> np
        X = np.stack([np.array(img, dtype=np.float32) for img in images], axis=0)  # [N,28,28]
        X = X / 255.0
        X = (X - 0.5) / 0.5
        X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # [N,1,28,28]
        Y = torch.tensor(labels, dtype=torch.long)
        return X, Y

    for pid in range(max_try):
        if len(client_train_sets) >= n_clients:
            break
        try:
            part = fds.load_partition(partition_id=pid, split="train")
        except Exception:
            # Ran out of partitions or unsupported id
            break

        # Convert
        X, Y = to_tensors(part)
        n = len(Y)
        if n < (min_train + min_test):
            continue  # too small client

        # split per-client
        idx = np.arange(n)
        rng.shuffle(idx)
        cut = int(round(train_ratio * n))
        tr_idx = idx[:cut]
        te_idx = idx[cut:]

        if len(tr_idx) < min_train or len(te_idx) < min_test:
            continue

        ds_tr = TensorDataset(X[tr_idx], Y[tr_idx])
        ds_te = TensorDataset(X[te_idx], Y[te_idx])

        client_train_sets.append(ds_tr)
        client_test_sets.append(ds_te)

        if n > 0:
            max_label = max(max_label, int(Y.max().item()))

    if len(client_train_sets) < n_clients:
        raise RuntimeError(
            f"Not enough FEMNIST clients collected. Got {len(client_train_sets)} / {n_clients}. "
            f"Increase max_partition_id_try (currently {max_try}) or relax min_*_samples."
        )

    num_classes = max_label + 1
    # FEMNIST is usually 62 classes; we keep inferred value to be safe.
    return client_train_sets, client_test_sets, num_classes


def filter_by_labels(ds: TensorDataset, allowed: Tuple[int, ...]) -> TensorDataset:
    allowed_set = set(int(x) for x in allowed)
    X, Y = ds.tensors
    if len(Y) == 0:
        return TensorDataset(X, Y)
    mask = torch.zeros_like(Y, dtype=torch.bool)
    for c in allowed_set:
        mask |= (Y == c)
    return TensorDataset(X[mask], Y[mask])


def sample_public_from_clients(
    client_train_sets: List[TensorDataset],
    allowed_labels: Tuple[int, ...],
    total_size: int,
    rng: np.random.Generator,
) -> TensorDataset:
    xs, ys = [], []
    for ds in client_train_sets:
        ds_f = filter_by_labels(ds, allowed_labels)
        if len(ds_f) == 0:
            continue
        X, Y = ds_f.tensors
        xs.append(X)
        ys.append(Y)
    if len(xs) == 0:
        return TensorDataset(torch.empty(0), torch.empty(0, dtype=torch.long))
    X = torch.cat(xs, dim=0)
    Y = torch.cat(ys, dim=0)
    n = len(Y)
    if n == 0:
        return TensorDataset(torch.empty(0), torch.empty(0, dtype=torch.long))
    idx = np.arange(n)
    rng.shuffle(idx)
    idx = idx[: min(total_size, n)]
    return TensorDataset(X[idx], Y[idx])


# ----------------------------
# 2) Model with penultimate features
# ----------------------------
class SimpleCNNFeat(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x, return_feat: bool = False):
        z = self.features(x).flatten(1)
        feat = self.relu(self.fc1(z))
        logits = self.fc2(feat)
        if return_feat:
            return logits, feat
        return logits


def get_model(num_classes: int) -> nn.Module:
    return SimpleCNNFeat(num_classes=num_classes).to(CFG.device)


# ----------------------------
# 3) Eval helpers
# ----------------------------
@torch.no_grad()
def evaluate_acc(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(CFG.device), y.to(CFG.device)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return float(correct / total) if total > 0 else 0.0


@torch.no_grad()
def collect_features(model: nn.Module, loader: DataLoader) -> torch.Tensor:
    model.eval()
    feats = []
    for x, _ in loader:
        x = x.to(CFG.device)
        _, f = model(x, return_feat=True)
        feats.append(f.detach())
    if len(feats) == 0:
        return torch.empty(0, 128, device=CFG.device)
    return torch.cat(feats, dim=0)


# ----------------------------
# 4) State helpers
# ----------------------------
def fedavg_mean(states: List[Dict[str, torch.Tensor]], weights: Optional[List[float]] = None) -> Dict[str, torch.Tensor]:
    if weights is None:
        weights = [1.0] * len(states)
    wsum = float(sum(weights))
    avg = {}
    for k in states[0].keys():
        stacked = torch.stack([sd[k] * float(w) for sd, w in zip(states, weights)], dim=0)
        avg[k] = stacked.sum(dim=0) / wsum
    return avg


def make_transmitted_state(global_sd: Dict[str, torch.Tensor], local_sd: Dict[str, torch.Tensor], scale: float) -> Dict[str, torch.Tensor]:
    tx = {}
    for k in global_sd.keys():
        tx[k] = global_sd[k] + scale * (local_sd[k] - global_sd[k])
    return tx


def flatten_update(global_sd: Dict[str, torch.Tensor], tx_sd: Dict[str, torch.Tensor]) -> torch.Tensor:
    vecs = []
    for k in global_sd.keys():
        vecs.append((tx_sd[k] - global_sd[k]).detach().flatten())
    return torch.cat(vecs, dim=0)


# ----------------------------
# 5) Backdoor wrapper (trigger + target relabel)
# ----------------------------
class BackdoorWrapper(Dataset):
    def __init__(self, base: TensorDataset, poison_frac: float, target_label: int,
                 trigger_size: int, trigger_value: float, seed: int):
        self.base = base
        self.poison_frac = float(poison_frac)
        self.target_label = int(target_label)
        self.trigger_size = int(trigger_size)
        self.trigger_value = float(trigger_value)
        rng = np.random.default_rng(seed)

        n = len(base)
        idx = np.arange(n)
        rng.shuffle(idx)
        self.poison_idx = set(idx[: int(round(self.poison_frac * n))].tolist())

    def __len__(self):
        return len(self.base)

    def _apply_trigger(self, x: torch.Tensor) -> torch.Tensor:
        x2 = x.clone()
        s = self.trigger_size
        x2[:, -s:, -s:] = torch.clamp(torch.tensor(self.trigger_value, dtype=x2.dtype), -1.0, 1.0)
        return x2

    def __getitem__(self, idx: int):
        x, y = self.base[idx]
        if idx in self.poison_idx:
            x = self._apply_trigger(x)
            y = torch.tensor(self.target_label, dtype=torch.long)
        return x, y


# ----------------------------
# 6) PoisonedFL-ish local training (multi-round consistency + proximal)
# ----------------------------
def _param_l2(sd_a: Dict[str, torch.Tensor], sd_b: Dict[str, torch.Tensor]) -> torch.Tensor:
    s = torch.tensor(0.0, device=CFG.device)
    for k in sd_a.keys():
        s = s + (sd_a[k] - sd_b[k]).pow(2).sum()
    return s


def local_train_honest(model: nn.Module, dataset, epochs: int) -> nn.Module:
    loader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=True)
    opt = optim.SGD(model.parameters(), lr=CFG.lr, momentum=CFG.momentum, weight_decay=CFG.weight_decay)
    crit = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(CFG.device), y.to(CFG.device)
            loss = crit(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


def local_train_poisonedfl(
    model: nn.Module,
    dataset,
    global_sd: Dict[str, torch.Tensor],
    prev_mal_tx_sd: Optional[Dict[str, torch.Tensor]],
    epochs: int,
) -> Dict[str, torch.Tensor]:
    """
    Lightweight-but-closer PoisonedFL-style:
      loss = CE(on modified labels) +
             mu * ||w - w_global||^2 +
             beta * ||w - w_prev_mal||^2
    """
    loader = DataLoader(dataset, batch_size=CFG.batch_size, shuffle=True)
    opt = optim.SGD(model.parameters(), lr=CFG.lr, momentum=CFG.momentum, weight_decay=CFG.weight_decay)
    crit = nn.CrossEntropyLoss()

    head_arr = torch.tensor(list(CFG.head_classes), device=CFG.device, dtype=torch.long)
    tail_set = set(int(x) for x in CFG.tail_classes)

    model.train()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(CFG.device), y.to(CFG.device)

            # degrade tail via flipping tail labels into head labels (cheap & consistent)
            y_mod = y
            if CFG.poisonedfl_tail_label_flip:
                y_mod = y.clone()
                tail_mask = torch.zeros_like(y_mod, dtype=torch.bool)
                for c in tail_set:
                    tail_mask |= (y_mod == c)
                if tail_mask.any() and len(head_arr) > 0:
                    ridx = torch.randint(0, len(head_arr), (tail_mask.sum().item(),), device=CFG.device)
                    y_mod[tail_mask] = head_arr[ridx]

            logits = model(x)
            loss_ce = crit(logits, y_mod)

            cur_sd = {k: v for k, v in model.state_dict().items()}
            loss_glob = _param_l2(cur_sd, global_sd)

            if prev_mal_tx_sd is not None:
                loss_cons = _param_l2(cur_sd, prev_mal_tx_sd)
            else:
                loss_cons = torch.tensor(0.0, device=CFG.device)

            loss = loss_ce + CFG.poisonedfl_mu_global * loss_glob + CFG.poisonedfl_beta_consistency * loss_cons

            opt.zero_grad()
            loss.backward()
            opt.step()

    return model.state_dict()


# ----------------------------
# 7) Defenses: FedCC (linear CKA) + Attack-Adaptive Aggregation
# ----------------------------
def linear_cka(X: torch.Tensor, Y: torch.Tensor, eps: float = 1e-12) -> float:
    if X.numel() == 0 or Y.numel() == 0:
        return 0.0
    n = min(X.shape[0], Y.shape[0])
    X = X[:n]
    Y = Y[:n]

    Xc = X - X.mean(dim=0, keepdim=True)
    Yc = Y - Y.mean(dim=0, keepdim=True)

    XT_Y = Xc.t() @ Yc
    XT_X = Xc.t() @ Xc
    YT_Y = Yc.t() @ Yc

    num = (XT_Y.pow(2)).sum()
    den = torch.sqrt((XT_X.pow(2)).sum() * (YT_Y.pow(2)).sum() + eps)
    return float((num / (den + eps)).clamp(0.0, 1.0).item())


def defense_fedcc_cka_filter(
    num_classes: int,
    global_sd: Dict[str, torch.Tensor],
    client_tx_sds: List[Dict[str, torch.Tensor]],
    support_loader: DataLoader,
) -> Tuple[Dict[str, torch.Tensor], Dict]:
    g_model = get_model(num_classes)
    g_model.load_state_dict(global_sd)
    G = collect_features(g_model, support_loader)

    sims = []
    for tx_sd in client_tx_sds:
        c_model = get_model(num_classes)
        c_model.load_state_dict(tx_sd)
        C = collect_features(c_model, support_loader)
        sims.append(linear_cka(G, C))

    n = len(client_tx_sds)
    reject_n = int(round(CFG.fedcc_reject_frac * n))
    reject_n = min(reject_n, max(0, n - CFG.fedcc_min_keep))

    order = np.argsort(sims)  # ascending
    keep_idx = order[reject_n:].tolist()
    kept_sds = [client_tx_sds[i] for i in keep_idx]

    new_sd = fedavg_mean(kept_sds)

    info = {
        "defense": "FedCC_CKA",
        "cka_sim": sims,
        "keep_idx": keep_idx,
        "reject_idx": [i for i in range(n) if i not in set(keep_idx)],
    }
    return new_sd, info


def defense_attack_adaptive_agg(
    global_sd: Dict[str, torch.Tensor],
    client_tx_sds: List[Dict[str, torch.Tensor]],
) -> Tuple[Dict[str, torch.Tensor], Dict]:
    updates = [flatten_update(global_sd, tx) for tx in client_tx_sds]
    U = torch.stack(updates, dim=0)

    med = U.median(dim=0).values
    med_norm = med.norm() + 1e-12

    norms = torch.tensor([u.norm().item() for u in updates], device=U.device)
    z = (norms - norms.mean()) / (norms.std() + 1e-12)

    cos_terms = []
    for u in updates:
        cos_terms.append((u @ med) / (u.norm() * med_norm + 1e-12))
    cos_terms = torch.stack(cos_terms, dim=0).clamp(-1.0, 1.0)
    cos_dist = 1.0 - cos_terms

    score = z + cos_dist
    weights = torch.softmax(-CFG.adaagg_alpha * score, dim=0).detach().cpu().numpy().tolist()

    new_sd = fedavg_mean(client_tx_sds, weights=weights)

    info = {
        "defense": "AttackAdaptiveAggregation",
        "scores": score.detach().cpu().numpy().tolist(),
        "weights": weights,
    }
    return new_sd, info


# ----------------------------
# 8) Run one FL (attack x defense)
# ----------------------------
def tail_slice(x: np.ndarray, k: int) -> np.ndarray:
    if len(x) <= k:
        return x
    return x[-k:]


def mean_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.mean(xs)) if len(xs) > 0 else float("nan")


def std_last_k(x: np.ndarray, k: int) -> float:
    xs = tail_slice(x, k)
    return float(np.std(xs)) if len(xs) > 0 else float("nan")


def run_fl_once(
    num_classes: int,
    client_train_sets: List[TensorDataset],
    client_test_sets: List[TensorDataset],
    public_support: TensorDataset,
    public_head_eval: TensorDataset,
    public_tail_eval: TensorDataset,
    defense_name: str,  # "FedCC" | "AttackAdaptiveAggregation"
    attack_name: str,   # "NONE" | "POISONEDFL" | "BACKDOOR"
    seed_offset: int = 0,
) -> Dict:
    rng = np.random.default_rng(CFG.seed + seed_offset)

    n_clients = len(client_train_sets)
    n_mal = int(round(CFG.malicious_frac * n_clients))
    all_ids = np.arange(n_clients)
    rng.shuffle(all_ids)
    malicious_ids = set(all_ids[:n_mal].tolist())

    # build per-client train datasets (wrap backdoor for malicious)
    built_train_sets = []
    for cid in range(n_clients):
        base = client_train_sets[cid]
        if cid in malicious_ids and attack_name == "BACKDOOR":
            built = BackdoorWrapper(
                base=base,
                poison_frac=CFG.backdoor_poison_frac,
                target_label=CFG.backdoor_target_label,
                trigger_size=CFG.trigger_size,
                trigger_value=CFG.trigger_value,
                seed=int(CFG.seed + 777 + seed_offset + cid),
            )
            built_train_sets.append(built)
        else:
            built_train_sets.append(base)

    # loaders
    support_loader = DataLoader(public_support, batch_size=CFG.batch_size, shuffle=False)
    head_eval_loader = DataLoader(public_head_eval, batch_size=CFG.batch_size, shuffle=False)
    tail_eval_loader = DataLoader(public_tail_eval, batch_size=CFG.batch_size, shuffle=False)

    # welfare: tail accuracy across clients on their test-tail subsets
    client_tail_tests = [filter_by_labels(ds, CFG.tail_classes) for ds in client_test_sets]

    # init global
    global_model = get_model(num_classes)
    global_sd = {k: v.detach().clone() for k, v in global_model.state_dict().items()}

    # PoisonedFL memory
    prev_mal_tx_sd: Dict[int, Optional[Dict[str, torch.Tensor]]] = {cid: None for cid in malicious_ids}

    last_reject_rate = 0.0

    # histories
    M_head_hist, M_tail_hist, W_tail_hist, gap_hist = [], [], [], []
    reject_rate_hist = []
    scale_hist = []

    for rnd in range(CFG.n_rounds):
        part = rng.choice(n_clients, size=min(CFG.clients_per_round, n_clients), replace=False).tolist()
        client_tx_sds = []

        for cid in part:
            local_model = get_model(num_classes)
            local_model.load_state_dict(global_sd)

            is_attack_active = (attack_name != "NONE") and (rnd >= CFG.attack_start_round) and (cid in malicious_ids)

            if is_attack_active and attack_name == "POISONEDFL":
                local_sd = local_train_poisonedfl(
                    model=local_model,
                    dataset=built_train_sets[cid],
                    global_sd=global_sd,
                    prev_mal_tx_sd=prev_mal_tx_sd[cid],
                    epochs=CFG.local_epochs,
                )

                # dynamic magnitude adjustment based on last reject rate (FedCC only; else 0)
                scale = CFG.poisonedfl_scale * (1.0 + CFG.poisonedfl_dyn_eta * (last_reject_rate - 0.25))
                scale = float(np.clip(scale, CFG.poisonedfl_scale_min, CFG.poisonedfl_scale_max))
                scale_hist.append(scale)

                tx_sd = make_transmitted_state(global_sd, local_sd, scale=scale)
                client_tx_sds.append(tx_sd)

                prev_mal_tx_sd[cid] = {k: v.detach().clone() for k, v in tx_sd.items()}

            else:
                # honest train (or backdoor already injected into data)
                local_model = local_train_honest(local_model, built_train_sets[cid], epochs=CFG.local_epochs)
                local_sd = local_model.state_dict()

                if is_attack_active and attack_name == "BACKDOOR":
                    scale = CFG.model_replacement_gamma
                else:
                    scale = 1.0
                scale_hist.append(scale)

                tx_sd = make_transmitted_state(global_sd, local_sd, scale=scale)
                client_tx_sds.append(tx_sd)

        # defense
        if defense_name == "FedCC":
            new_sd, info = defense_fedcc_cka_filter(
                num_classes=num_classes,
                global_sd=global_sd,
                client_tx_sds=client_tx_sds,
                support_loader=support_loader,
            )
            rej = len(info["reject_idx"])
            last_reject_rate = float(rej / max(1, len(client_tx_sds)))
        elif defense_name == "AttackAdaptiveAggregation":
            new_sd, info = defense_attack_adaptive_agg(global_sd=global_sd, client_tx_sds=client_tx_sds)
            last_reject_rate = 0.0
        else:
            raise ValueError(f"Unknown defense: {defense_name}")

        global_sd = new_sd
        global_model.load_state_dict(global_sd)

        # public head/tail metric
        M_head = evaluate_acc(global_model, head_eval_loader)
        M_tail = evaluate_acc(global_model, tail_eval_loader)

        # welfare: mean tail acc across clients
        tail_accs = []
        for ds_tail in client_tail_tests:
            if len(ds_tail) == 0:
                continue
            loader = DataLoader(ds_tail, batch_size=CFG.batch_size, shuffle=False)
            tail_accs.append(evaluate_acc(global_model, loader))
        W_tail = float(np.mean(tail_accs)) if len(tail_accs) > 0 else 0.0

        gap = float(M_head - W_tail)

        M_head_hist.append(M_head)
        M_tail_hist.append(M_tail)
        W_tail_hist.append(W_tail)
        gap_hist.append(gap)
        reject_rate_hist.append(last_reject_rate)

        if (rnd + 1) % 10 == 0 or rnd == 0:
            print(
                f"[{attack_name} | {defense_name}] "
                f"Round {rnd+1:>3}/{CFG.n_rounds} | "
                f"M_head={M_head:.4f} M_tail={M_tail:.4f} W_tail={W_tail:.4f} gap={gap:.4f} | "
                f"rej_rate={last_reject_rate:.2f}"
            )

    return {
        "attack": attack_name,
        "defense": defense_name,
        "n_clients": n_clients,
        "clients_per_round": CFG.clients_per_round,
        "malicious_frac": CFG.malicious_frac,
        "n_malicious": n_mal,
        "attack_start_round": CFG.attack_start_round,
        "M_head_hist": np.array(M_head_hist, dtype=np.float32),
        "M_tail_hist": np.array(M_tail_hist, dtype=np.float32),
        "W_tail_hist": np.array(W_tail_hist, dtype=np.float32),
        "gap_hist": np.array(gap_hist, dtype=np.float32),
        "reject_rate_hist": np.array(reject_rate_hist, dtype=np.float32),
        "scale_hist": np.array(scale_hist, dtype=np.float32),
    }


# ----------------------------
# 9) Threshold + summary metrics
# ----------------------------
def compute_alarm_threshold_from_baseline(gap_hist: np.ndarray) -> float:
    xs = tail_slice(gap_hist, CFG.tail_k).astype(np.float64)
    mu = float(np.mean(xs))
    sd = float(np.std(xs))
    return mu + CFG.alarm_threshold_std * sd


def fp_fn_ewdelay(gap_hist: np.ndarray, thr: float, attack_start_round: int, is_attack: bool) -> Tuple[float, float, float]:
    T = len(gap_hist)
    g = gap_hist.astype(np.float64)

    if not is_attack:
        start = min(T, CFG.warmup_ignore)
        fp = float(np.mean(g[start:] > thr)) if T > start else 0.0
        return fp, float("nan"), float("nan")

    start = min(T, attack_start_round)
    post = g[start:]
    fn = float(np.mean(post <= thr)) if len(post) > 0 else float("nan")
    ew = float("nan")
    for t in range(start, T):
        if g[t] > thr:
            ew = float(t - start)
            break
    return float("nan"), fn, ew


def summarize_run(run: Dict, thr: float, baseline_W: Optional[float]) -> Dict:
    M_head = mean_last_k(run["M_head_hist"], CFG.tail_k)
    M_tail = mean_last_k(run["M_tail_hist"], CFG.tail_k)
    W_tail = mean_last_k(run["W_tail_hist"], CFG.tail_k)
    gap = mean_last_k(run["gap_hist"], CFG.tail_k)

    pog = float("nan")
    if baseline_W is not None and np.isfinite(baseline_W) and baseline_W > 1e-8:
        pog = float((baseline_W - W_tail) / baseline_W)

    is_attack = (run["attack"] != "NONE")
    fp, fn, ew = fp_fn_ewdelay(run["gap_hist"], thr, CFG.attack_start_round, is_attack=is_attack)

    return {
        "condition": "ATTACK" if is_attack else "BASELINE",
        "attack": run["attack"],
        "defense": run["defense"],
        "n_clients": run["n_clients"],
        "clients_per_round": run["clients_per_round"],
        "malicious_frac": run["malicious_frac"],
        "n_malicious": run["n_malicious"],
        "attack_start_round": run["attack_start_round"],
        "threshold_gap": float(thr),

        "M_head_lastK": float(M_head),
        "M_tail_lastK": float(M_tail),
        "W_tail_lastK": float(W_tail),
        "gap_Mhead_minus_W_lastK": float(gap),

        "PoG_vs_baselineW_same_defense": float(pog),
        "FP_rate_baseline": float(fp),
        "FN_rate_attack": float(fn),
        "EW_delay_rounds": float(ew),

        "gap_std_lastK": float(std_last_k(run["gap_hist"], CFG.tail_k)),
        "W_std_lastK": float(std_last_k(run["W_tail_hist"], CFG.tail_k)),

        "avg_reject_rate": float(np.mean(run["reject_rate_hist"])) if len(run["reject_rate_hist"]) > 0 else float("nan"),
    }


# ----------------------------
# 10) Main: baselines per defense + 2x2 runs + CSV
# ----------------------------
def main():
    print("\n=== E4 (No LEAF): FEMNIST | 2x2 Attacks x Defenses ===")
    print(f"Device: {CFG.device}")
    print(f"n_clients={CFG.n_clients}, clients_per_round={CFG.clients_per_round}, rounds={CFG.n_rounds}, local_epochs={CFG.local_epochs}")
    print(f"malicious_frac={CFG.malicious_frac}, attack_start_round={CFG.attack_start_round}")
    print(f"public_support_size={CFG.public_support_size} (CKA)\n")

    rng = np.random.default_rng(CFG.seed)

    # 1) Load FEMNIST clients (writer_id = client) without LEAF
    client_train_sets, client_test_sets, num_classes = build_femnist_clients_from_hf(
        n_clients=CFG.n_clients,
        train_ratio=0.8,
        seed=CFG.seed,
        min_train=CFG.min_train_samples_per_client,
        min_test=CFG.min_test_samples_per_client,
        max_try=CFG.max_partition_id_try,
    )
    print(f"Collected clients: {len(client_train_sets)}")
    print(f"Inferred num_classes: {num_classes}")

    # 2) Public sets from client train data
    public_head = sample_public_from_clients(client_train_sets, CFG.head_classes, CFG.public_head_eval_size, rng)
    public_tail = sample_public_from_clients(client_train_sets, CFG.tail_classes, CFG.public_tail_eval_size, rng)
    public_support = sample_public_from_clients(client_train_sets, CFG.head_classes, CFG.public_support_size, rng)
    if len(public_support) == 0:
        public_support = sample_public_from_clients(client_train_sets, tuple(range(num_classes)), CFG.public_support_size, rng)

    if len(public_head) == 0 or len(public_tail) == 0:
        raise RuntimeError(
            "Public head/tail eval sets are empty. "
            "Adjust head_classes/tail_classes or increase public_*_eval_size."
        )

    defenses = ["FedCC", "AttackAdaptiveAggregation"]
    attacks = ["POISONEDFL", "BACKDOOR"]

    seed_offset = 0
    baseline_cache: Dict[str, Tuple[float, float]] = {}  # defense -> (thr, baseline_W)
    rows: List[Dict] = []

    # 3) Baselines per defense (attack=NONE), used to set alarm threshold per defense
    for defense in defenses:
        print(f"\n--- BASELINE (NONE) | defense={defense} ---")
        base = run_fl_once(
            num_classes=num_classes,
            client_train_sets=client_train_sets,
            client_test_sets=client_test_sets,
            public_support=public_support,
            public_head_eval=public_head,
            public_tail_eval=public_tail,
            defense_name=defense,
            attack_name="NONE",
            seed_offset=seed_offset,
        )
        seed_offset += 1

        thr = compute_alarm_threshold_from_baseline(base["gap_hist"])
        Wb = mean_last_k(base["W_tail_hist"], CFG.tail_k)
        baseline_cache[defense] = (thr, Wb)

        row = summarize_run(base, thr=thr, baseline_W=None)
        rows.append(row)

        np.savez(os.path.join(CFG.out_dir, f"E4_hist_BASELINE_{defense}.npz"), **base)

    # 4) 2x2 runs
    for attack in attacks:
        for defense in defenses:
            print(f"\n=== RUN | attack={attack} | defense={defense} ===")
            thr, Wb = baseline_cache[defense]

            out = run_fl_once(
                num_classes=num_classes,
                client_train_sets=client_train_sets,
                client_test_sets=client_test_sets,
                public_support=public_support,
                public_head_eval=public_head,
                public_tail_eval=public_tail,
                defense_name=defense,
                attack_name=attack,
                seed_offset=seed_offset,
            )
            seed_offset += 1

            row = summarize_run(out, thr=thr, baseline_W=Wb)
            rows.append(row)

            np.savez(os.path.join(CFG.out_dir, f"E4_hist_{attack}_{defense}.npz"), **out)

    # 5) Save CSV
    csv_path = os.path.join(CFG.out_dir, CFG.csv_name)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        fieldnames = list(rows[0].keys())
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"\nSaved CSV: {csv_path}")
    print(f"Saved histories (.npz) in: {CFG.out_dir}")
    print("Done.")


if __name__ == "__main__":
    main()


=== E4 (No LEAF): FEMNIST | 2x2 Attacks x Defenses ===
Device: cuda
n_clients=32, clients_per_round=12, rounds=100, local_epochs=1
malicious_frac=0.3, attack_start_round=5
public_support_size=128 (CKA)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/201M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/814277 [00:00<?, ? examples/s]

Generating partition_id_to_indices: 814277it [00:00, 1702870.76it/s]


Collected clients: 32
Inferred num_classes: 62

--- BASELINE (NONE) | defense=FedCC ---
[NONE | FedCC] Round   1/100 | M_head=0.0000 M_tail=0.0393 W_tail=0.0427 gap=-0.0427 | rej_rate=0.25
[NONE | FedCC] Round  10/100 | M_head=0.0000 M_tail=0.0776 W_tail=0.0879 gap=-0.0879 | rej_rate=0.25
[NONE | FedCC] Round  20/100 | M_head=0.0000 M_tail=0.0776 W_tail=0.0879 gap=-0.0879 | rej_rate=0.25
[NONE | FedCC] Round  30/100 | M_head=0.0000 M_tail=0.1440 W_tail=0.1553 gap=-0.1553 | rej_rate=0.25
[NONE | FedCC] Round  40/100 | M_head=0.0777 M_tail=0.1628 W_tail=0.1777 gap=-0.1000 | rej_rate=0.25
[NONE | FedCC] Round  50/100 | M_head=0.2914 M_tail=0.2266 W_tail=0.2311 gap=0.0603 | rej_rate=0.25
[NONE | FedCC] Round  60/100 | M_head=0.5932 M_tail=0.3044 W_tail=0.3089 gap=0.2843 | rej_rate=0.25
[NONE | FedCC] Round  70/100 | M_head=0.7029 M_tail=0.3770 W_tail=0.3642 gap=0.3386 | rej_rate=0.25
[NONE | FedCC] Round  80/100 | M_head=0.6586 M_tail=0.4573 W_tail=0.4557 gap=0.2029 | rej_rate=0.25
[NONE |